# Solar Filament V27 — FilamentFormer-Lite (instance-query learning)

**A new trainable instance-segmentation architecture**, not another YOLO threshold sweep. Trains a compact convolutional feature pyramid + learned Transformer instance queries; uses *separate* filament polygon masks, Hungarian assignment, mask Dice/BCE, expert-annotated spine/centerline, boundary, and orientation supervision, and held-date quality-controlled fusion with the **confirmed V25 public PQ 0.39**.

**Kaggle inputs:** official `solar3k` MAGFiLO dataset and `SOLAR_V25_ALL_TRAINED_CHECKPOINTS.zip` **or** its extracted models (V20 global/native `.pt`, V24 refiner `.pt`, V25 real refiner `.pt`, V25 curve `.pt`, V25 quality `.joblib`). To preserve byte-identical public 0.39, also attach `SOLAR_V25_ALL_PREDICTIONS_AND_REPORTS.zip`, or the original V25 winning `V25_REAL_REFINE_PLUS_QUALITY.csv`. V25 trained-model full ZIP may embed the CSV archive already.

GPU T4/T4×2 (one GPU used), enable internet only for missing packages. Checkpointed training and prediction. **The 104 V20 validation images were reused in earlier experiments; date-disjoint audit here is exploratory, not an untouched test set.** Do not train on hidden test annotations. Any 0.40 is a *target*, not a claim.

**Design:** A network can output multiple distinct instance masks even when their foreground areas touch. V25 still supplies the known-high-precision masks, and novel V27 detections must earn their place on date-disjoint local PQ. Synthetic sanity tests and checks are included; full GPU performance is not known until you run this notebook.

In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); OUT=Path('/kaggle/working/solar_v27')
SEED=2026; GPU_ID=0; H=W=2048
RAW_CONF=.08; RAW_IOU=.72; MAX_DET=120
ALLOW_NON_V20_WEIGHT_HASH=False; REQUIRE_V24_MODELS=True
V25_EXPECTED_QUALITY_CUTOFF=.38
V25_EXPECTED_CSV_SHA='ac470fbfa0e2ca5f2d'; V25_EXPECTED_ROWS=1093
CROP_SIDE=320
CURVE_WINDOW=768; CURVE_STRIDE=640; CURVE_SIDE=512
CURVE_MIN_AREA=170; CURVE_MAX_PROPOSALS=42
# Query model: 512px input, 128px mask lattice; native crop covers 768x768.
QUERY_CROP_NATIVE=768; QUERY_SIDE=512; QUERY_MASK_SIDE=128
QUERY_COUNT=24; MAX_TARGET_INSTANCES=24
WINDOW_STRIDE=640; CURVE_AUX_WEIGHT=1.0
TRAIN_CROPS_PER_SCENE=4; EPOCHS=11; STEPS_PER_EPOCH=185; BATCH=2
TRAIN_STRUCTURE_MODEL=True
TRAIN_PLAIN_ABLATION=True  # toggle False if compute constrained; training only the main model is enough to submit
PLAIN_EPOCHS=5; PLAIN_STEPS=125
TRAINING_LR=2.2e-4; WARM_START=None
# New candidate inference: independent masks + per-query confidence and learned quality
RAW_QUERY_CONF=.34; QUERY_MASK_PROB=.48
MIN_QUERY_AREA=170; QUERY_MAX_PER_SCENE=55
QUERY_NMS_IOU=.37; QUERY_TILE_SEAM_GUARD=True
QUALITY_CANDIDATE_THRESHOLDS=[.20,.30,.40,.50,.60,.70]
FUSION_MIN_FREE=.72; FUSION_NMS_IOU=.23
MIN_LOCAL_GAIN_TO_PROMOTE=.004
# Stored artifacts are reusable if added to Kaggle input on a subsequent session.
for sub in ('models','raw','refined','curve_raw','reports','submissions','query','query_cache','input_data'):
    (OUT/sub).mkdir(parents=True,exist_ok=True)
assert INPUT.is_dir(), 'Run on Kaggle with the official solar3k and V25 trained model datasets.'
print('V27 root:',OUT)


In [ ]:
import os,sys,time,json,hashlib,zipfile,shutil,gzip,random,subprocess,importlib.util
import gc as _gc_safe
from collections import defaultdict
import numpy as np,pandas as pd,cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from IPython.display import display

def require(module,package):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
require('pycocotools','pycocotools')
# Lock exact V22 version when available to maximize baseline reproduction.
try:
    from importlib.metadata import version as package_version
    installed_version=package_version('ultralytics')
except Exception:
    installed_version=None
if installed_version!='8.4.189':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.189'])
from pycocotools import mask as coco
from ultralytics import YOLO
import ultralytics,torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU T4/T4x2. No CPU or nano fallback.'
assert torch.cuda.device_count()>GPU_ID
print('Using',torch.cuda.get_device_name(GPU_ID),'ultralytics',ultralytics.__version__)
cv2.setNumThreads(0);random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
H=W=2048
EXT={'.jpeg','.jpg','.png','.tif','.tiff','.bmp'}
def files_in(d):
    p=Path(d)
    return sorted(q for q in p.iterdir() if q.is_file() and q.suffix.lower() in EXT) if p.is_dir() else []
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def save_json(p,obj):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_suffix(p.suffix+'.tmp');temp.write_text(json.dumps(obj,indent=2,default=str));temp.replace(p)
def enc(m):return coco.encode(np.asfortranarray(np.asarray(m,np.uint8)))
def txt(r):return r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else str(r['counts'])
def as_rle(x):return {'size':[H,W],'counts':x.encode('ascii') if isinstance(x,str) else x}
assert callable(_gc_safe.collect)
save_json(OUT/'reports'/'env.json',{'torch':torch.__version__,'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(GPU_ID),'no_prior_split_required':True,'prior_verified_public_PQ':0.39,'target_public_PQ':0.40})

## 1. Official data and the historical V20 acquisition-day partition

In [ ]:
ANAME='MAGFiLO_1.0_Annotations_kaggle2026_train.json'
found=sorted(INPUT.rglob(ANAME))
OFFICIAL=next((p.parent.parent for p in found if (p.parent.parent/'test'/'test_images').is_dir() and (p.parent.parent/'train'/'train_images').is_dir()),None)

def find_zip(name):
    matches=list(INPUT.rglob(name))
    return matches[0] if matches else None

def safe_images_unzip(archive,destination):
    destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if not item.filename.lower().endswith(tuple(EXT)):continue
            target=destination/Path(item.filename).name
            if not target.exists():
                with z.open(item) as src,open(target,'wb') as dst:shutil.copyfileobj(src,dst,1<<20)
    return destination
if OFFICIAL:
    ANNOT=OFFICIAL/'train'/ANAME
    TRAIN={p.stem:p for p in files_in(OFFICIAL/'train'/'train_images')}
    TEST={p.stem:p for p in files_in(OFFICIAL/'test'/'test_images')}
    print('Using full official competition folder',OFFICIAL)
else:
    annot=found[0] if found else None
    ztest=find_zip('test.zip');ztrain=find_zip('sometrain.zip')
    assert annot is not None and ztest is not None, 'Attach official solar3k, or the annotation JSON and test.zip as Kaggle inputs.'
    ANNOT=annot
    TEST={p.stem:p for p in files_in(safe_images_unzip(ztest,OUT/'input_data'/'test_images'))}
    TRAIN={p.stem:p for p in files_in(safe_images_unzip(ztrain,OUT/'input_data'/'train_images'))} if ztrain else {}
    print('Using upload ZIP fallback. Validation may cover fewer than 104 physical images.')
assert len(TEST)==180,('Expected 180 test scenes; found',len(TEST))
assert len(set(TEST)&set(TRAIN))==0
DOC=json.loads(ANNOT.read_text())
BY_ID=defaultdict(list)
for ann in DOC['annotations']:BY_ID[str(ann['image_id'])].append(ann)
VERSIONS=defaultdict(list)
for item in DOC['images']:
    stem=Path(item['file_name']).stem
    if stem in TRAIN:VERSIONS[stem].append((str(item['id']),BY_ID.get(str(item['id']),[])))
assert set(TRAIN)==set(VERSIONS)
if TRAIN:
    days=defaultdict(list)
    for img in DOC['images']:
        stem=Path(img['file_name']).stem
        days[stem[:8]].append(stem)
    days={k:sorted(set(v)) for k,v in days.items()}
    shuffled=sorted(days)
    random.Random(SEED+2020).shuffle(shuffled)
    selected_days=[];n=0
    for d in shuffled:
        if n>=104:break
        selected_days.append(d);n+=len(days[d])
    VALIDATION=sorted(set(stem for d in selected_days for stem in days[d]) & set(TRAIN))
else:VALIDATION=[]
FULL_V20_VALIDATION=(len(VALIDATION)==104 and len(TRAIN)==707)
print('Images: train present',len(TRAIN),'test',len(TEST),'V20 tune available',len(VALIDATION),'full reproduction',FULL_V20_VALIDATION)
if VALIDATION:
    assert not ({s[:8] for s in VALIDATION} & {s[:8] for s in TRAIN if s not in VALIDATION})
save_json(OUT/'reports'/'dataset.json',{'annotation_sha256':sha(ANNOT),'train_available':len(TRAIN),
    'test_images':len(TEST),'validation_available':len(VALIDATION),'full_v20_reproduction':FULL_V20_VALIDATION,
    'validation_stems':VALIDATION,'test_ground_truth_used':False})

## 2. Restore verified frozen V20 detectors

In [ ]:
EXPECT={
 'portable_global_m':('portable_global_m_best.pt','73fd3f3a4931'),
 'portable_native_m':('portable_native_m_best.pt','1a932d28e590'),
}
V20_DIR=Path('/kaggle/input/datasets/lighittha/solarouts/solar_v20_from_scratch/models')
ZIPS=sorted(set(p for p in INPUT.rglob('*.zip') if any(token in p.name.lower() for token in ('v20','v21','weights','models'))))
WEIGHTS={};HASHES={};SOURCES={}
(OUT/'models').mkdir(parents=True,exist_ok=True)
for model_name,(filename,prefix) in EXPECT.items():
    path=OUT/'models'/filename
    choices=[]
    direct=V20_DIR/filename
    if direct.is_file():choices.append(direct)
    choices+=[p for p in INPUT.rglob(filename) if p.is_file() and p!=direct]
    if path.is_file():choices.insert(0,path)
    if not path.is_file() or path.stat().st_size<20_000_000:
        restored=False
        for src in choices:
            if 30_000_000<src.stat().st_size<65_000_000:
                if src!=path:shutil.copy2(src,path)
                SOURCES[model_name]=str(src);restored=True;break
        if not restored:
            for archive in ZIPS:
                try:
                    with zipfile.ZipFile(archive) as z:
                        member='models/'+filename
                        if member not in z.namelist():continue
                        if not (30_000_000<z.getinfo(member).file_size<65_000_000):continue
                        temp=path.with_suffix('.part')
                        with z.open(member) as rf,open(temp,'wb') as wf:shutil.copyfileobj(rf,wf,1<<20)
                        temp.replace(path);SOURCES[model_name]=str(archive)+'::'+member
                        restored=True;break
                except (OSError,zipfile.BadZipFile) as e:print('Ignore corrupt ZIP:',archive,e)
        if not restored:raise FileNotFoundError('Missing trained V20 '+filename+f'. Expected at {V20_DIR/filename} or inside the V20 weights ZIP. This notebook never substitutes pretrained-only or nano models.')
    digest=sha(path)
    if not ALLOW_NON_V20_WEIGHT_HASH:
        assert digest.startswith(prefix),f'{filename}: wrong weights hash! Expected V20 prefix {prefix}, got {digest[:12]}'
    m=YOLO(str(path))
    n_params=sum(p.numel() for p in m.model.parameters())
    assert m.task=='segment' and 21_000_000<=n_params<=24_000_000,(model_name,m.task,n_params)
    del m;_gc_safe.collect()
    WEIGHTS[model_name]=path;HASHES[model_name]=digest
print('Recovered V20 model identity:',{k:v[:12] for k,v in HASHES.items()})
save_json(OUT/'reports'/'weights.json',{'sha256':HASHES,'source':SOURCES,'V20_checkpoint_hash_gate':not ALLOW_NON_V20_WEIGHT_HASH})

## 3. Immutable V22 detector proposal generation and original mask selection

In [ ]:
# Locked V22 successful source definitions.
SRC_BASE_GLOBAL=('portable_global_m','g1536')
SRC_BASE_NATIVE=('portable_native_m','g1536')
SRC_HI=('portable_global_m','g1792')
SRC_TILES=('portable_native_m','t1280')
SRC_HIFLIP=('portable_global_m','g1792_hflip')
BASE_SOURCES=[SRC_HI,SRC_TILES]
EXTRA_SOURCES=[]
ACTIVE=BASE_SOURCES+EXTRA_SOURCES
ENGINES={}
def get_engine(name):
    if name not in ENGINES:
        ENGINES.clear();_gc_safe.collect();torch.cuda.empty_cache()
        ENGINES[name]=YOLO(str(WEIGHTS[name]))
    return ENGINES[name]

def view_windows(view):
    if view=='g1536':return [(0,0,2048,1536,False)]
    if view=='g1792':return [(0,0,2048,1792,False)]
    if view=='g1792_hflip':return [(0,0,2048,1792,True)]
    if view=='t1280':return [(x,y,1280,1024,False) for y in (0,768) for x in (0,768)]
    raise ValueError(view)
def sig(source):
    name,view=source
    return hashlib.sha256(json.dumps({'model_sha256':HASHES[name],'view':view,
      'conf':RAW_CONF,'iou':RAW_IOU,'max_det':MAX_DET,'retina_masks':True,
      'agnostic_nms':True,'ultralytics':ultralytics.__version__,'version':'V22_1' if view in ('g1792','t1280','g1536') else 'V24_1'},sort_keys=True).encode()).hexdigest()
def cache_file(source,stem):
    name,view=source
    return OUT/'raw'/name/view/(stem+'.json.gz')
@torch.inference_mode()
def raw_predictions(source,stem):
    path=cache_file(source,stem);identifier=sig(source)
    if path.exists():
        try:
            with gzip.open(path,'rt') as f:v=json.load(f)
            if v.get('signature')==identifier and v.get('stem')==stem:return v['predictions']
        except (OSError,ValueError,KeyError):pass
    image_path=TRAIN.get(stem) or TEST.get(stem)
    if image_path is None:raise FileNotFoundError('Missing scene: '+stem)
    gray=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
    assert gray is not None and gray.shape==(H,W),(stem, None if gray is None else gray.shape)
    predictions=[]
    name,view=source
    model=get_engine(name)
    for x,y,side,net,hflip in view_windows(view):
        roi=np.ascontiguousarray(gray[y:y+side,x:x+side])
        if hflip:roi=cv2.flip(roi,1)
        image=cv2.cvtColor(roi,cv2.COLOR_GRAY2BGR)
        result=model.predict(image,imgsz=net,conf=RAW_CONF,iou=RAW_IOU,max_det=MAX_DET,
                             agnostic_nms=True,retina_masks=True,device=GPU_ID,verbose=False)[0]
        if result.masks is None or result.boxes is None:continue
        mask_data=result.masks.data.detach().cpu().numpy()
        confidences=result.boxes.conf.detach().cpu().numpy()
        for mask_arr,conf in zip(mask_data,confidences):
            mask=(mask_arr>0.5).astype(np.uint8)
            if mask.shape!=(side,side):mask=cv2.resize(mask,(side,side),interpolation=cv2.INTER_NEAREST)
            if hflip:mask=np.ascontiguousarray(cv2.flip(mask,1))
            if int(mask.sum())<100:continue
            if side!=2048:
                ys,xs=np.where(mask)
                truncated=bool((x>0 and xs.min()<10) or (x+side<W and xs.max()>=side-10) or
                               (y>0 and ys.min()<10) or (y+side<H and ys.max()>=side-10))
                full=np.zeros((H,W),np.uint8)
                full[y:y+side,x:x+side]=mask
            else:truncated=False;full=mask
            rr=enc(full)
            predictions.append({'counts':txt(rr),'score':float(conf),'area':int(coco.area(rr)),
                                'truncated':truncated,'source_view':view})
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with gzip.open(tmp,'wt',compresslevel=3) as f:
        json.dump({'signature':identifier,'stem':stem,'view':view,'predictions':predictions},f,separators=(',',':'))
    tmp.replace(path)
    return predictions
print('V23 views including immutable winner:',ACTIVE)

In [ ]:
def recipe(name,sources,gconf=.24,tconf=.32,area=200,seam=False,refine='none',minfree=.45):
    return {'name':name,'sources':[list(s) for s in sources],'global_conf':float(gconf),
            'tile_conf':float(tconf),'min_area':int(area),'seam_guard':bool(seam),
            'nms_iou':.50,'tile_bonus':.08,'min_free':float(minfree),'refine':refine}

# This exact configuration produced V22_ALT_2.csv (public PQ 0.37).
WINNER=recipe('V22_ALT2_WINNER_037',[SRC_HI,SRC_TILES],
              gconf=.24,tconf=.42,area=200,seam=True,refine='none',minfree=.45)
WINNER['nms_iou']=.50
WINNER['tile_bonus']=.08
WINNER['engine']='v22'
WINNER['fusion']='none'
WINNER['tile_policy']='baseline'

def experiment(name,sources=None,gconf=.24,tconf=.42,area=200,seam=True,
               bonus=.08,nms=.50,minfree=.45,tile_policy='baseline',fusion='none'):
    r=recipe(name,sources if sources is not None else [SRC_HI,SRC_TILES],
             gconf=gconf,tconf=tconf,area=area,seam=seam,minfree=minfree)
    r.update(engine='v23',tile_bonus=bonus,nms_iou=nms,tile_policy=tile_policy,fusion=fusion)
    return r


def modify_rle(rr,operation):
    if operation=='none':return rr
    box=coco.toBbox(rr);x,y,w,h=[int(q) for q in box]
    if w<1 or h<1:return rr
    pad=3;x0=max(0,x-pad);y0=max(0,y-pad);x1=min(W,x+w+pad);y1=min(H,y+h+pad)
    full=coco.decode(rr).astype(np.uint8)
    roi=full[y0:y1,x0:x1]
    if operation=='close3':
        result=cv2.morphologyEx(roi,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
    elif operation=='dilate1':
        result=cv2.dilate(roi,np.ones((3,3),np.uint8),iterations=1)
    elif operation=='erode1':
        result=cv2.erode(roi,np.ones((3,3),np.uint8),iterations=1)
    else:raise ValueError(operation)
    if int(result.sum())<50:return rr
    full[y0:y1,x0:x1]=result
    return enc(full)

def select_masks(raw,cfg):
    sources={tuple(pair) for pair in cfg['sources']}
    candidates=[]
    for source,items in raw.items():
        if source not in sources:continue
        threshold=cfg['tile_conf'] if source[1].startswith('t') else cfg['global_conf']
        for p in items:
            score=float(p['score']);area=int(p['area'])
            if score<threshold or area<cfg['min_area']:continue
            if cfg['seam_guard'] and source[1].startswith('t') and p.get('truncated',False):continue
            rank=score+(cfg['tile_bonus'] if source[1].startswith('t') else 0.)
            candidates.append((rank,area,p))
    candidates.sort(key=lambda q:(-q[0],-q[1]))
    masks=[];originals=[];occupied=np.zeros((H,W),np.uint8)
    for rank,area,p in candidates:
        rr=as_rle(p['counts'])
        if originals:
            ov=np.asarray(coco.iou([rr],originals,[0]*len(originals)))[0]
            if np.any(ov>=cfg['nms_iou']):continue
        mask=coco.decode(rr).astype(np.uint8)
        free=mask&(occupied==0)
        free_area=int(free.sum())
        if free_area<cfg['min_area'] or free_area/max(1,area)<cfg['min_free']:continue
        cleaned=enc(free)
        if cfg.get('refine','none')!='none':cleaned=modify_rle(cleaned,cfg['refine'])
        # Retain exact V20 occupied behavior for baseline; for refined recipes use actual refined footprint.
        if cfg.get('refine','none')!='none':
            shaped=coco.decode(cleaned).astype(np.uint8)&(occupied==0)
            if int(shaped.sum())<cfg['min_area']:continue
            cleaned=enc(shaped)
            occupied|=shaped
        else:occupied|=free
        masks.append(cleaned);originals.append(rr)
        if len(masks)>=120:break
    return masks

# Do not change the winner branch; these tests protect the baseline.
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['global_conf']==.24 and WINNER['tile_conf']==.42
assert WINNER['seam_guard'] and WINNER['min_area']==200
assert WINNER['nms_iou']==.5 and WINNER['tile_bonus']==.08
assert callable(_gc_safe.collect)

## 4. Local expert-version PQ proxy (same definition as V25)

In [ ]:
def annotation_rle(a):
    seg=a.get('segmentation')
    if seg is None or seg==[]:return None
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):
            obj=coco.frPyObjects(seg,H,W)
            return coco.merge(obj) if isinstance(obj,list) else obj
        rr=dict(seg)
        if isinstance(rr.get('counts'),str):rr['counts']=rr['counts'].encode('ascii')
        return rr
    if len(seg)>0 and isinstance(seg[0],(int,float)):seg=[seg]
    polygons=[p for p in seg if len(p)>=6 and len(p)%2==0]
    if not polygons:return None
    rr=coco.frPyObjects(polygons,H,W)
    return coco.merge(rr) if isinstance(rr,list) else rr
TRUTH={}
def gt(stem):
    if stem not in TRUTH:
        TRUTH[stem]=[[rr for a in arr if (rr:=annotation_rle(a)) is not None]
                     for _,arr in VERSIONS[stem]]
    return TRUTH[stem]
def match_counts(truth,preds):
    if not truth:return np.array([0.,0.,float(len(preds)),0.])
    if not preds:return np.array([0.,0.,0.,float(len(truth))])
    ious=np.asarray(coco.iou(truth,preds,[0]*len(preds)),float)
    # Match the earlier V20 PQ proxy EXACTLY for control-score parity.
    ii,jj=linear_sum_assignment(-ious)
    ok=ious[ii,jj]>.5
    tp=int(ok.sum())
    return np.array([float(ious[ii[ok],jj[ok]].sum()),float(tp),
                     float(len(preds)-tp),float(len(truth)-tp)])
def physical_counts(stem,preds):
    result=np.zeros(4,float)
    for version in gt(stem):result+=match_counts(version,preds)
    return result
def pq(count):
    den=count[1]+.5*(count[2]+count[3])
    return float(count[0]/den) if den>0 else 0.
def metrics(count):
    den=count[1]+.5*(count[2]+count[3])
    return {'PQ':pq(count),'SQ':float(count[0]/count[1]) if count[1] else 0.,
       'RQ':float(count[1]/den) if den else 0.,'TP':int(count[1]),
       'FP':int(count[2]),'FN':int(count[3])}
example=np.zeros((H,W),np.uint8);example[40:61,50:92]=1
assert coco.area(as_rle(txt(enc(example))))==example.sum()
assert abs(pq(match_counts([enc(example)],[enc(example)]))-1)<1e-9
print('PQ scorer and full-size RLE roundtrip: PASS')

## 5. Recover trained V24 and V25 model components; preserve the real 0.39 pipeline

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import joblib, math, time, io
from torchvision.models import resnet18

class DoubleConv(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.block=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
            nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x):return self.block(x)

class PromptUNet(nn.Module):
    def __init__(self):
        super().__init__();e=resnet18(weights=None);old=e.conv1
        e.conv1=nn.Conv2d(4,64,7,2,3,bias=False)
        self.stem=nn.Sequential(e.conv1,e.bn1,e.relu);self.pool=e.maxpool
        self.e1=e.layer1;self.e2=e.layer2;self.e3=e.layer3;self.e4=e.layer4
        self.d4=DoubleConv(768,192);self.d3=DoubleConv(320,128)
        self.d2=DoubleConv(192,96);self.d1=DoubleConv(160,64)
        self.out=nn.Sequential(nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                               nn.Conv2d(64,32,3,padding=1),nn.ReLU(inplace=True),nn.Conv2d(32,1,1))
    def forward(self,x):
        e0=self.stem(x);e1=self.e1(self.pool(e0));e2=self.e2(e1);e3=self.e3(e2);e4=self.e4(e3)
        def up(a,b):return torch.cat([F.interpolate(a,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1)
        a=self.d4(up(e4,e3));a=self.d3(up(a,e2));a=self.d2(up(a,e1));a=self.d1(up(a,e0))
        return self.out(a)

def make_channels(gray_uint8,mask_uint8):
    gray=np.asarray(gray_uint8,dtype=np.uint8)
    equal=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(gray)
    blur=cv2.GaussianBlur(gray,(3,3),0)
    dx=cv2.Sobel(blur,cv2.CV_32F,1,0,ksize=3);dy=cv2.Sobel(blur,cv2.CV_32F,0,1,ksize=3)
    edge=np.clip(cv2.magnitude(dx,dy)/240.,0,1)
    return np.stack([gray/255.,equal/255.,mask_uint8.astype(np.float32),edge],axis=0).astype(np.float32)

def normalized_bbox_roi(box,scale=1.48,min_extent=120):
    x,y,w,h=map(float,box)
    if w<=0 or h<=0:return None
    dx=min(W,max(float(min_extent),w*scale));dy=min(H,max(float(min_extent),h*scale))
    cx=x+w/2;cy=y+h/2
    x0=int(np.clip(round(cx-dx/2),0,W-1));y0=int(np.clip(round(cy-dy/2),0,H-1))
    x1=int(np.clip(round(cx+dx/2),x0+1,W));y1=int(np.clip(round(cy+dy/2),y0+1,H))
    return x0,y0,x1,y1

def find_original_asset(filename):
    direct=sorted(INPUT.rglob(filename))
    for p in direct:
        if p.is_file() and p.stat().st_size>100:return p.read_bytes(),str(p)
    for z in sorted(INPUT.rglob('*.zip')):
        if not any(token in z.name.lower() for token in ('v24','model','trained')):continue
        try:
            with zipfile.ZipFile(z) as arc:
                matches=[m for m in arc.namelist() if Path(m).name==filename]
                if matches:return arc.read(matches[0]),str(z)+'::'+matches[0]
                # V24 full output bundles a second ZIP containing its scored CSVs.
                for nested in [m for m in arc.namelist() if m.lower().endswith('.zip')]:
                    try:
                        with zipfile.ZipFile(io.BytesIO(arc.read(nested))) as inner:
                            hits=[x for x in inner.namelist() if Path(x).name==filename]
                            if hits:return inner.read(hits[0]),str(z)+'::'+nested+'::'+hits[0]
                    except (OSError,zipfile.BadZipFile):pass
        except (OSError,zipfile.BadZipFile):continue
    return None,None

for filename in ('SOLAR_V24_REFINER_BEST.pt','SOLAR_V24_INSTANCE_QUALITY.joblib'):
    target=OUT/'models'/filename
    data,source=find_original_asset(filename)
    if data is None:
        if REQUIRE_V24_MODELS:raise FileNotFoundError('Attach V24 model archive; missing '+filename)
    else:
        if not target.exists() or sha(target)!=hashlib.sha256(data).hexdigest():target.write_bytes(data)
        print('Recovered V24 asset:',filename, 'from',source,'SHA:',sha(target)[:16])

V24_REFINER_PATH=OUT/'models'/'SOLAR_V24_REFINER_BEST.pt'
V24_QUALITY_PATH=OUT/'models'/'SOLAR_V24_INSTANCE_QUALITY.joblib'
refiner_v24=PromptUNet().to('cuda:'+str(GPU_ID))
refiner_v24.load_state_dict(torch.load(V24_REFINER_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_v24.eval()
meta_v24=joblib.load(V24_QUALITY_PATH)
quality_v24=meta_v24['model'];V24_FEATURES=meta_v24['features']
assert len(V24_FEATURES)==15 and callable(quality_v24.predict_proba)
print('Recovered genuine V24 quality model, OOF AUROC:',meta_v24.get('oof_auc'))

In [ ]:
def matching_scores(rr,records):
    if not records:return 0.,0.,0.
    rles=[as_rle(q['counts']) for q in records]
    ious=np.asarray(coco.iou([rr],rles,[0]*len(rles)),float).reshape(-1)
    i=int(ious.argmax());return float(ious[i]),float(records[i]['score']),float(max(q['score'] for q in records))

@torch.inference_mode()
def run_refiner(model,gray,rr,threshold=.50,min_prompt_iou=.54):
    crop=normalized_bbox_roi([int(v) for v in coco.toBbox(rr)],scale=1.48)
    if crop is None:return rr
    x0,y0,x1,y1=crop;original=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    if not original.any():return rr
    image=cv2.resize(gray[y0:y1,x0:x1],(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_AREA)
    prompt=cv2.resize(original,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
    inp=torch.from_numpy(make_channels(image,prompt))[None].cuda(GPU_ID)
    with torch.autocast('cuda',dtype=torch.float16):
        probabilities=model(inp).sigmoid()[0,0].float().cpu().numpy()
    refined=(cv2.resize(probabilities,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)>=threshold).astype(np.uint8)
    oa=int(original.sum());na=int(refined.sum())
    if na<80 or not (.52<=na/max(oa,1)<=1.65):return rr
    overlap=int((refined&original).sum())/max(1,oa+na-int((refined&original).sum()))
    if overlap<min_prompt_iou:return rr
    full=np.zeros((H,W),np.uint8);full[y0:y1,x0:x1]=refined
    return enc(full)

def load_base(stem):
    raw={SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    return raw,select_masks(raw,WINNER)

def v24_refined_masks(stem,baseline,gray=None):
    key=sha(V24_REFINER_PATH)[:18];p=OUT/'refined'/('v24_'+key+'_'+stem+'.json.gz')
    if p.is_file():
        with gzip.open(p,'rt') as f:v=json.load(f)
        if len(v)==len(baseline):return [as_rle(x) for x in v]
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    proposed=[run_refiner(refiner_v24,gray,rr) for rr in baseline]
    occupied=np.zeros((H,W),np.uint8);final=[]
    for a,b in zip(baseline,proposed):
        f=coco.decode(b).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:f=coco.decode(a).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:continue
        final.append(enc(f));occupied|=f
    if len(final)!=len(baseline):final=baseline
    p.parent.mkdir(parents=True,exist_ok=True)
    with gzip.open(p,'wt',compresslevel=3) as f:json.dump([txt(v) for v in final],f)
    return final

def features_v24(stem,base,refined,gray=None,raw=None):
    raw=raw if raw is not None else {SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    rows=[]
    for rr,new in zip(base,refined):
        x,y,w,h=map(float,coco.toBbox(rr));mask=coco.decode(rr).astype(np.uint8);area=int(mask.sum())
        dilation=cv2.dilate(mask,np.ones((11,11),np.uint8))
        ring=(dilation>0)&(mask==0)
        inside=float(gray[mask>0].mean()) if area else 0.;outside=float(gray[ring].mean()) if ring.any() else inside
        ga,gscore,gmax=matching_scores(rr,raw[SRC_HI]);ta,tscore,tmax=matching_scores(rr,raw[SRC_TILES])
        na=float(coco.area(new));oa=float(coco.area(rr))
        riou=float(coco.iou([rr],[new],[0])[0,0])
        rows.append({'base_score':max(gscore,tscore),'base_area_log':float(np.log1p(area)),
                     'base_box_ar':float(np.log((w+1)/(h+1))),'base_fill':float(area/max(1,w*h)),
                     'base_center_x':float((x+w/2)/W),'base_center_y':float((y+h/2)/H),
                     'global_agreement':ga,'tile_agreement':ta,'global_maxconf':gmax,'tile_maxconf':tmax,
                     'pixel_avg_inside':inside/255.,'pixel_avg_ring':outside/255.,
                     'contrast':(outside-inside)/255.,'refined_iou':riou,'refined_area_ratio':na/max(oa,1.)})
    return pd.DataFrame(rows,columns=V24_FEATURES)

def v24_control(stem):
    raw,base=load_base(stem)
    if not base:return []
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=v24_refined_masks(stem,base,gray)
    df=features_v24(stem,base,refined,gray,raw)
    qs=quality_v24.predict_proba(df[V24_FEATURES].fillna(0.))[:,1]
    return [rr for rr,q in zip(base,qs) if q>=.25]  # Exact V24 QUALITY_025: no extra clipping

s=np.zeros((H,W),np.uint8);s[50:90,70:130]=1
assert int(coco.area(enc(s)))==2400
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['tile_conf']==.42 and WINNER['global_conf']==.24 and WINNER['seam_guard']
print('V24 control frozen. New V25 scoring cannot alter its thresholds or masks.')

In [ ]:
# Restore all genuinely trained V25 checkpoint bytes; no substitute/pretrained-only networks.
from skimage.morphology import skeletonize
from skimage.segmentation import watershed

V25_REAL_PATH=OUT/'models'/'SOLAR_V25_REAL_REFINER_BEST.pt'
V25_CURVE_PATH=OUT/'models'/'SOLAR_V25_CURVE_BEST.pt'
V25_QUALITY_PATH=OUT/'models'/'SOLAR_V25_EXTENDED_QUALITY.joblib'
for p in (V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH):
    data,source=find_original_asset(p.name)
    if data is None:raise FileNotFoundError('Missing '+p.name+'. Attach the complete V25 trained checkpoint ZIP.')
    if not p.exists() or sha(p)!=hashlib.sha256(data).hexdigest():p.write_bytes(data)
    print('Recovered frozen V25',p.name,sha(p)[:18], 'from',source)
refiner_real=PromptUNet().cuda(GPU_ID)
refiner_real.load_state_dict(torch.load(V25_REAL_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_real.eval();REAL_REFINER_SHA=sha(V25_REAL_PATH)

def real_refine(stem,rr,gray=None):
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    return run_refiner(refiner_real,gray,rr,threshold=.5,min_prompt_iou=.50)

v25_ranker_package=joblib.load(V25_QUALITY_PATH)
new_rankers=v25_ranker_package['models']
V25_QUALITY_CUTOFF=float(v25_ranker_package['cutoff'])
print('Frozen V25 ranker cutoff:',V25_QUALITY_CUTOFF,'feature count:',len(v25_ranker_package['columns']))
assert abs(V25_QUALITY_CUTOFF-V25_EXPECTED_QUALITY_CUTOFF)<1e-6, 'Wrong V25 ranking checkpoint/cutoff'

def rank_probs(models,df):
    if df.empty:return np.asarray([],float)
    a=df[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)
    return np.mean([m.predict_proba(a)[:,1] for m in models],axis=0)


In [ ]:
class MiniUNet(nn.Module):
    def __init__(self,channels=2,outputs=3):
        super().__init__()
        self.a=DoubleConv(channels,24);self.b=DoubleConv(24,48);self.c=DoubleConv(48,96)
        self.d=DoubleConv(96,160)
        self.up3=DoubleConv(160+96,96);self.up2=DoubleConv(96+48,64)
        self.up1=DoubleConv(64+24,32);self.final=nn.Conv2d(32,outputs,1)
    def forward(self,x):
        a=self.a(x);b=self.b(F.avg_pool2d(a,2));c=self.c(F.avg_pool2d(b,2));d=self.d(F.avg_pool2d(c,2))
        u=self.up3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        u=self.up2(torch.cat([F.interpolate(u,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        u=self.up1(torch.cat([F.interpolate(u,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return self.final(u)


curve_path=V25_CURVE_PATH
curve_net=MiniUNet().cuda(GPU_ID)
curve_net.load_state_dict(torch.load(curve_path,map_location='cpu',weights_only=False)['model'],strict=True)
curve_net.eval();CURVE_SHA=sha(curve_path)
print('Loaded frozen V25 curve network',CURVE_SHA[:18])

In [ ]:
def curve_locations():
    starts=list(range(0,H-CURVE_WINDOW+1,CURVE_STRIDE))
    if starts[-1]!=H-CURVE_WINDOW:starts.append(H-CURVE_WINDOW)
    return [(x,y) for y in starts for x in starts]
CURVE_LOCATIONS=curve_locations()
assert len(CURVE_LOCATIONS)==9, CURVE_LOCATIONS

def curve_input(gray):
    equ=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(gray)
    return torch.from_numpy(np.stack([gray/255.,equ/255.],0).astype(np.float32))[None].cuda(GPU_ID)

@torch.inference_mode()
def curve_proposals(stem):
    cache=OUT/'curve_raw'/(stem+'.json.gz')
    signature=hashlib.sha256((CURVE_SHA+stem+str(CURVE_WINDOW)+str(CURVE_SIDE)).encode()).hexdigest()
    if cache.exists():
        try:
            with gzip.open(cache,'rt') as f:v=json.load(f)
            if v.get('identity')==signature:return v['proposals']
        except (ValueError,OSError,KeyError):pass
    image=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    if image is None:raise FileNotFoundError(stem)
    props=[]
    for x0,y0 in CURVE_LOCATIONS:
        raw=image[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]
        inp=cv2.resize(raw,(CURVE_SIDE,CURVE_SIDE),interpolation=cv2.INTER_AREA)
        with torch.autocast('cuda',dtype=torch.float16):
            probs=curve_net(curve_input(inp)).sigmoid()[0].float().cpu().numpy()
        fg=cv2.resize(probs[0],(CURVE_WINDOW,CURVE_WINDOW))
        center=cv2.resize(probs[1],(CURVE_WINDOW,CURVE_WINDOW))
        seg=(fg>.53).astype(np.uint8)
        n,components,stats,_=cv2.connectedComponentsWithStats(seg,connectivity=8)
        for k in range(1,n):
            component_area=int(stats[k,cv2.CC_STAT_AREA])
            if component_area<CURVE_MIN_AREA:continue
            local=(components==k).astype(np.uint8)
            # Conservatively split only when two *distinct* centerline islands exist.
            line=(center>.56).astype(np.uint8)&local
            m,markers,mark_stats,_=cv2.connectedComponentsWithStats(line,connectivity=8)
            good=[q for q in range(1,m) if mark_stats[q,cv2.CC_STAT_AREA]>=8]
            pieces=[local]
            if 2<=len(good)<=5 and component_area>700:
                seeds=np.zeros_like(local,np.int32)
                for label_idx,mark_idx in enumerate(good,1):seeds[markers==mark_idx]=label_idx
                regions=watershed(-cv2.distanceTransform(local,cv2.DIST_L2,3),seeds,mask=(local>0))
                candidates=[(regions==z).astype(np.uint8) for z in range(1,len(good)+1)]
                if all(int(c.sum())>=CURVE_MIN_AREA for c in candidates):pieces=candidates
            for part in pieces:
                area=int(part.sum())
                if area<CURVE_MIN_AREA:continue
                sy,sx=np.where(part)
                if min(sx)<8 and x0>0 or max(sx)>CURVE_WINDOW-9 and x0+CURVE_WINDOW<W:continue
                if min(sy)<8 and y0>0 or max(sy)>CURVE_WINDOW-9 and y0+CURVE_WINDOW<H:continue
                local_gray=raw[part>0];mask_score=float(np.mean(fg[part>0]))
                line_score=float(np.mean(center[part>0]))
                if mask_score<.59 or line_score<.18:continue
                # Relatively dark features, not arbitrary bright solar active-region structures.
                patch=raw[max(0,sy.min()-16):min(CURVE_WINDOW,sy.max()+17),
                          max(0,sx.min()-16):min(CURVE_WINDOW,sx.max()+17)]
                if len(local_gray) and float(np.median(local_gray))>float(np.percentile(patch,75))+8:continue
                full=np.zeros((H,W),np.uint8)
                full[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]=part
                rr=enc(full)
                props.append({'counts':txt(rr),'score':float(np.clip(.65*mask_score+.35*line_score,0,1)),
                              'area':int(coco.area(rr)),'truncated':False,'source_view':'curve_v25',
                              'center_support':line_score})
    props.sort(key=lambda p:p['score'],reverse=True)
    distinct=[];rles=[]
    for p in props:
        rr=as_rle(p['counts'])
        if rles and float(np.max(coco.iou([rr],rles,[0]*len(rles))))>.33:continue
        distinct.append(p);rles.append(rr)
        if len(distinct)>=CURVE_MAX_PROPOSALS:break
    with gzip.open(cache,'wt',compresslevel=3) as f:
        json.dump({'identity':signature,'stem':stem,'proposals':distinct},f,separators=(',',':'))
    return distinct

def extend_with_curve(stem,baseline,raw=None):
    # Existing instance identities are never silently split or replaced here.
    raw=raw if raw is not None else load_base(stem)[0]
    base_rles=list(baseline)
    base_all=base_rles.copy()
    occupied=np.zeros((H,W),np.uint8)
    for rr in base_rles:occupied|=coco.decode(rr).astype(np.uint8)
    additions=[]
    for p in curve_proposals(stem):
        rr=as_rle(p['counts'])
        if base_all and float(np.max(coco.iou([rr],base_all,[0]*len(base_all))))>=.25:continue
        part=coco.decode(rr).astype(np.uint8)
        free=part&(occupied==0)
        if int(free.sum())<CURVE_MIN_AREA or free.sum()/max(1,part.sum())<.70:continue
        accepted=enc(free)
        additions.append(accepted);occupied|=free;base_all.append(accepted)
    return base_rles+additions,['detector']*len(base_rles)+['curve']*len(additions)

print('Curve candidate generation initialized:',len(CURVE_LOCATIONS),'windows per 2048 scene')

In [ ]:
from scipy.spatial.distance import cdist

# These 40+ new features are computed from *predicted masks/images only*.
EXTRA_FIELDS=['log_perimeter','log_convex_area','solidity','extent','elongation','eccentricity',
 'aspect_absolute','area_frac_box','edge_density','endpoint_count','branchpoint_count','skeleton_length',
 'skeleton_area_ratio','mean_thickness','max_thickness','width_variation','local_component_count',
 'intensity_inside_std','intensity_inside_p10','intensity_inside_p50','intensity_inside_p90',
 'intensity_ring_p50','intensity_diff_p10','intensity_diff_mean','inside_entropy','sobel_inside_mean',
 'sobel_ring_mean','contrast_normalized','sun_radial','rim_distance','mask_near_solar_limb',
 'hull_vs_mask','compactness','hu_1','hu_2','pca_angle_sin','pca_angle_cos',
 'neighbor_center_dist','neighbor_bbox_iou','neighbor_area_ratio','n_neighbors_100',
 'curve_origin','curve_conf','curve_center_support','curve_vs_global_iou','curve_vs_tile_iou',
 'raw_global_count','raw_tile_count','raw_high_conf_global_count','raw_high_conf_tile_count',
 'mask_global_rank','mask_tile_rank','mask_area_over_disk','intensity_outside_std']
NEW_FEATURES=list(V24_FEATURES)+EXTRA_FIELDS
assert len(NEW_FEATURES)>60

def image_entropy(pixels):
    if len(pixels)==0:return 0.
    hist=np.bincount((pixels//16).astype(np.uint8),minlength=16).astype(float)
    hist=hist/max(hist.sum(),1);h=hist[hist>0]
    return float(-(h*np.log(h)).sum())

def extract_extra(gray,rr,all_boxes,source_kind,curve_meta,raw):
    x,y,w,h=map(float,coco.toBbox(rr));x0=max(0,int(x)-12);y0=max(0,int(y)-12)
    x1=min(W,int(np.ceil(x+w))+12);y1=min(H,int(np.ceil(y+h))+12)
    # COCO decode is full-frame but geometric operations are limited to a small crop.
    mask=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    g=gray[y0:y1,x0:x1];area=int(mask.sum());yy,xx=np.where(mask>0)
    ring=cv2.dilate(mask,np.ones((9,9),np.uint8))&(1-mask)
    inside=g[mask>0];outside=g[ring>0]
    if not len(inside):inside=np.asarray([0],np.uint8)
    if not len(outside):outside=inside
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    perimeter=float(sum(cv2.arcLength(c,True) for c in contours))
    hull_area=float(sum(cv2.contourArea(cv2.convexHull(c)) for c in contours if len(c)>=3))
    # Downsample skeleton for speed; normalize pixel-to-image ratio.
    ratio=min(1.,150/max(mask.shape));sw=max(8,int(mask.shape[1]*ratio));sh=max(8,int(mask.shape[0]*ratio))
    reduced=(cv2.resize(mask,(sw,sh),interpolation=cv2.INTER_NEAREST)>0).astype(np.uint8)
    sk=skeletonize(reduced>0).astype(np.uint8)
    neigh=cv2.filter2D(sk,cv2.CV_16S,np.ones((3,3),np.uint8),borderType=cv2.BORDER_CONSTANT)-sk.astype(np.int16)
    endpoints=int(((sk>0)&(neigh==1)).sum());branches=int(((sk>0)&(neigh>=3)).sum())
    dist=cv2.distanceTransform(reduced,cv2.DIST_L2,3)[reduced>0]
    components=cv2.connectedComponents(reduced,connectivity=8)[0]-1
    if len(xx)>1:
        cov=np.cov(np.stack([xx,yy]).astype(float));ev,vec=np.linalg.eigh(cov)
        major=vec[:,np.argmax(ev)];elong=float(np.sqrt((max(ev)+1)/(min(ev)+1)))
        ecc=float(np.sqrt(max(0.,1-min(ev)/(max(ev)+1e-6))))
    else:major=np.array([1.,0.]);elong=1.;ecc=0.
    sob=cv2.magnitude(cv2.Sobel(g,cv2.CV_32F,1,0),cv2.Sobel(g,cv2.CV_32F,0,1))
    hu=cv2.HuMoments(cv2.moments(mask)).flatten()
    cx=x+w/2;cy=y+h/2;rad=np.hypot(cx-W/2,cy-H/2)/(W/2)
    other_boxes=np.asarray(all_boxes,float).reshape(-1,4)
    current_centroid=np.array([cx,cy]);centroids=other_boxes[:,:2]+other_boxes[:,2:]/2 if len(other_boxes) else np.zeros((0,2))
    distances=np.linalg.norm(centroids-current_centroid,axis=1) if len(centroids) else np.asarray([])
    nearby=sorted(float(z) for z in distances if z>1)
    bbox_ious=[]
    for bx,by,bw,bh in other_boxes:
        if abs(bx-x)+abs(by-y)+abs(bw-w)+abs(bh-h)<.001:continue
        iw=max(0,min(x+w,bx+bw)-max(x,bx));ih=max(0,min(y+h,by+bh)-max(y,by))
        bbox_ious.append(iw*ih/max(1,w*h+bw*bh-iw*ih))
    raw_g=raw[SRC_HI];raw_t=raw[SRC_TILES]
    gi,gc,gmax=matching_scores(rr,raw_g);ti,tc,tmax=matching_scores(rr,raw_t)
    flat=dict(log_perimeter=np.log1p(perimeter),log_convex_area=np.log1p(hull_area),
        solidity=area/max(1,hull_area),extent=area/max(1,w*h),elongation=min(100,elong),
        eccentricity=ecc,aspect_absolute=max((w+1)/(h+1),(h+1)/(w+1)),area_frac_box=area/max(1,w*h),
        edge_density=perimeter/max(1,area),endpoint_count=endpoints,branchpoint_count=branches,
        skeleton_length=int(sk.sum())/max(.15,ratio),skeleton_area_ratio=float(sk.sum())/max(1,reduced.sum()),
        mean_thickness=float(np.mean(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        max_thickness=float(np.max(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        width_variation=float(np.std(dist)) if len(dist) else 0.,local_component_count=max(0,components),
        intensity_inside_std=float(inside.std())/255.,intensity_inside_p10=float(np.percentile(inside,10))/255.,
        intensity_inside_p50=float(np.median(inside))/255.,intensity_inside_p90=float(np.percentile(inside,90))/255.,
        intensity_ring_p50=float(np.median(outside))/255.,
        intensity_diff_p10=float(np.percentile(outside,10)-np.percentile(inside,10))/255.,
        intensity_diff_mean=float(outside.mean()-inside.mean())/255.,inside_entropy=image_entropy(inside),
        sobel_inside_mean=float(sob[mask>0].mean())/255. if area else 0.,
        sobel_ring_mean=float(sob[ring>0].mean())/255. if ring.any() else 0.,
        contrast_normalized=float(outside.mean()-inside.mean())/max(10.,float(outside.std())),
        sun_radial=rad,rim_distance=max(0,1-rad),mask_near_solar_limb=float(rad>.93),
        hull_vs_mask=hull_area/max(1,area),compactness=perimeter**2/max(1,4*np.pi*area),
        hu_1=float(np.sign(hu[0])*np.log1p(abs(hu[0]))),hu_2=float(np.sign(hu[1])*np.log1p(abs(hu[1]))),
        pca_angle_sin=float(major[1]),pca_angle_cos=float(major[0]),
        neighbor_center_dist=nearby[0]/2048 if nearby else 1.,
        neighbor_bbox_iou=max(bbox_ious,default=0.),
        neighbor_area_ratio=float(sorted([abs(np.log((w*h+1)/(bw*bh+1))) for bx,by,bw,bh in other_boxes])[1]) if len(other_boxes)>1 else 0.,
        n_neighbors_100=sum(v<100 for v in nearby),curve_origin=float(source_kind=='curve'),
        curve_conf=float(curve_meta.get('score',0)),curve_center_support=float(curve_meta.get('center_support',0)),
        curve_vs_global_iou=gi,curve_vs_tile_iou=ti,
        raw_global_count=len(raw_g),raw_tile_count=len(raw_t),
        raw_high_conf_global_count=sum(p['score']>.4 for p in raw_g),
        raw_high_conf_tile_count=sum(p['score']>.4 for p in raw_t),
        mask_global_rank=gc,mask_tile_rank=tc,mask_area_over_disk=area/(np.pi*1024**2),
        intensity_outside_std=float(outside.std())/255.)
    assert set(flat)==set(EXTRA_FIELDS),('Missing feature',set(EXTRA_FIELDS)-set(flat))
    return flat

def extended_features(stem,masks,origins=None,curve_meta=None,gray=None,raw=None):
    if not masks:return pd.DataFrame(columns=NEW_FEATURES)
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    if raw is None:raw=load_base(stem)[0]
    # Refinement-based V24 features need the frozen, correct V24 refiner.
    if origins is None:origins=['detector']*len(masks)
    if curve_meta is None:curve_meta=[{} for _ in masks]
    assert len(masks)==len(origins)==len(curve_meta)
    refined=v24_refined_masks(stem,masks,gray) if all(o=='detector' for o in origins) else [
        run_refiner(refiner_v24,gray,r) if o=='detector' else r for r,o in zip(masks,origins)]
    base=features_v24(stem,masks,refined,gray,raw)
    boxes=[list(map(float,coco.toBbox(rr))) for rr in masks]
    for i,rr in enumerate(masks):
        extra=extract_extra(gray,rr,boxes,origins[i],curve_meta[i],raw)
        for k in EXTRA_FIELDS:base.loc[i,k]=extra[k]
    return base[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)

print('Extended feature count:',len(NEW_FEATURES))

In [ ]:
VAL_STEMS=sorted(VALIDATION)
assert len(VAL_STEMS)==104 or not FULL_VALIDATION_REQUIRED
by_day=defaultdict(list)
for st in VAL_STEMS:by_day[st[:8]].append(st)
all_days=sorted(by_day);random.Random(SEED+952).shuffle(all_days)
CAL_DAYS=set(all_days[:int(len(all_days)*.52)])
CAL=sorted(s for s in VAL_STEMS if s[:8] in CAL_DAYS)
AUDIT=sorted(s for s in VAL_STEMS if s[:8] not in CAL_DAYS)
assert len(CAL)>30 and len(AUDIT)>30 and not ({s[:8] for s in CAL}&{s[:8] for s in AUDIT})
print('Train quality on',len(CAL),'scenes, evaluate on',len(AUDIT),'different-day scenes')

# Cache masks and features separately from the checkpointed models.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:
        full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    meta=[]
    for index,rr in enumerate(full):
        if origins[index]=='detector':meta.append({})
        else:
            index_ious=np.asarray(coco.iou([rr],[as_rle(v['counts']) for v in curve_list],
                                             [0]*len(curve_list)),float).reshape(-1) if curve_list else np.asarray([])
            meta.append(curve_list[int(index_ious.argmax())] if len(index_ious) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    path=OUT/'reports'/('features_'+stem+'.joblib')
    # Data are caches of MODEL PREDICTIONS, not ground-truth test labels.
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if path.exists():
        try:
            obj=joblib.load(path)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    obj={'signature':signature,'masks':[txt(x) for x in masks], 'origins':origins,
         'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(obj,path,compress=3)
    return obj

def consensus_quality_labels(stem,preds):
    expert_versions=gt(stem);out=[]
    for rr in preds:
        votes=[]
        for truth in expert_versions:
            if truth:
                vec=np.asarray(coco.iou([rr],truth,[0]*len(truth)),float).reshape(-1)
                votes.append(float(np.max(vec)) if len(vec) else 0.)
            else:votes.append(0.)
        out.append(int(np.mean(np.asarray(votes)>.5)>=.5) if votes else 0)
    return out


assert list(v25_ranker_package['columns'])==NEW_FEATURES,'V25 feature order mismatch'


In [ ]:
# Use all frozen V25 components to reconstruct its exact candidate-order/quality behavior.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    curve_rles=[as_rle(p['counts']) for p in curve_list]
    meta=[]
    for i,rr in enumerate(full):
        if origins[i]=='detector':meta.append({});continue
        overlaps=np.asarray(coco.iou([rr],curve_rles,[0]*len(curve_rles)),float).reshape(-1) if curve_rles else np.asarray([])
        meta.append(curve_list[int(overlaps.argmax())] if len(overlaps) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    location=OUT/'reports'/('v25_features_'+stem+'.joblib')
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+sha(V25_QUALITY_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if location.is_file():
        try:
            obj=joblib.load(location)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    result={'signature':signature,'masks':[txt(r) for r in masks], 'origins':origins,
            'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(result,location,compress=3)
    return result

def nonoverlap_after_refinement(masks):
    occupied=np.zeros((H,W),np.uint8);out=[]
    for rr in masks:
        pix=coco.decode(rr).astype(np.uint8)&(occupied==0)
        if int(pix.sum())<80:continue
        out.append(enc(pix));occupied|=pix
    return out

def frozen_v25_masks(stem, obj=None):
    info=obj if obj is not None else scene_feature_data(stem)
    prob=rank_probs(new_rankers,info['features'])
    inds=[i for i in range(info['n_base']) if prob[i]>=V25_QUALITY_CUTOFF]
    masks=[as_rle(info['masks'][i]) for i in inds]
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=[real_refine(stem,rr,gray) if prob[i]>=.50 else rr for i,rr in zip(inds,masks)]
    return nonoverlap_after_refinement(refined)

assert V25_QUALITY_CUTOFF==V25_EXPECTED_QUALITY_CUTOFF
assert list(v25_ranker_package['columns'])==NEW_FEATURES
print('V25 0.39 candidate logic restored, original ranker/curve/refiner frozen.')


## 6. Prepare independent polygon-instance training windows

We sample one complete annotator version per scene/window and preserve every instance as its **own mask**. No semantic union is used for Hungarian training. Only the historic **603 detector-training scenes** contribute gradient updates; the 104 reused validation scenes supply experiment diagnostics. Scene-grouped training/validation subsets are disjoint.

In [ ]:
import math
from skimage.morphology import skeletonize
from scipy.optimize import linear_sum_assignment
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

def masks_in_crop(annotations, x0, y0, side, target=QUERY_MASK_SIDE):
    masks=[];centers=[]
    scale=target/float(side)
    for a in annotations:
        bx,by,bw,bh=map(float,a.get('bbox',[0,0,0,0]))
        if bx+bw<x0 or by+bh<y0 or bx>x0+side or by>y0+side:continue
        seg=a.get('segmentation',[])
        if isinstance(seg,dict):
            rr=annotation_rle(a)
            if rr is None:continue
            sub=coco.decode(rr)[y0:y0+side,x0:x0+side].astype(np.uint8)
            small=cv2.resize(sub,(target,target),interpolation=cv2.INTER_NEAREST)
        else:
            if seg and isinstance(seg[0],(float,int)):seg=[seg]
            small=np.zeros((target,target),np.uint8)
            for polygon in seg:
                if len(polygon)<6 or len(polygon)%2:continue
                coords=np.asarray(polygon,dtype=np.float32).reshape(-1,2)
                pts=np.rint((coords-[x0,y0])*scale).astype(np.int32)
                cv2.fillPoly(small,[pts],1)
        area=int(small.sum())
        if area<6:continue
        masks.append(small.astype(np.uint8))
        centers.append((bx+bw/2-x0,by+bh/2-y0,area))
    if len(masks)>MAX_TARGET_INSTANCES:
        # Keep large prominent objects, with some shorter instances retained via a mixture.
        order=sorted(range(len(masks)),key=lambda i:-centers[i][2])
        selected=order[:MAX_TARGET_INSTANCES]
        masks=[masks[i] for i in selected]
    return np.stack(masks) if masks else np.zeros((0,target,target),np.uint8)

train_stems=sorted(set(TRAIN)-set(VALIDATION))
assert len(train_stems)==603 and not(set(train_stems)&set(VALIDATION))
train_day_map=defaultdict(list)
for st in train_stems:train_day_map[st[:8]].append(st)
all_train_days=sorted(train_day_map)
random.Random(SEED+427).shuffle(all_train_days)
inner_val_days=set(all_train_days[:max(12,round(.12*len(all_train_days)))])
TRAIN_INNER=[s for s in train_stems if s[:8] not in inner_val_days]
VAL_INNER=[s for s in train_stems if s[:8] in inner_val_days]
assert TRAIN_INNER and VAL_INNER
assert not ({s[:8] for s in TRAIN_INNER}&{s[:8] for s in VAL_INNER})

def build_records(stems,repeat):
    entries=[]
    for stem in stems:
        versions=[anns for _,anns in VERSIONS[stem] if len(anns)]
        if not versions:continue
        rng=random.Random(SEED+int(hashlib.sha256(stem.encode()).hexdigest()[:8],16))
        for k in range(repeat):
            ann=rng.choice(versions)
            anchor=rng.choice(ann)
            bx,by,bw,bh=map(float,anchor.get('bbox',[0,0,0,0]))
            cx=bx+bw/2+rng.uniform(-175,175)
            cy=by+bh/2+rng.uniform(-175,175)
            x0=int(np.clip(round(cx-QUERY_CROP_NATIVE/2),0,W-QUERY_CROP_NATIVE))
            y0=int(np.clip(round(cy-QUERY_CROP_NATIVE/2),0,H-QUERY_CROP_NATIVE))
            entries.append((stem,x0,y0,k))
    return entries
TRAIN_RECORDS=build_records(TRAIN_INNER,TRAIN_CROPS_PER_SCENE)
VAL_RECORDS=build_records(VAL_INNER,2)
assert len(TRAIN_RECORDS)>800 and len(VAL_RECORDS)>100
print('Instance-query training samples:',len(TRAIN_RECORDS),'validation:',len(VAL_RECORDS),
      'train acquisition-days:',len(set(s[:8] for s in TRAIN_INNER)))

def annotated_spine_map(annotations,x0,y0,side,target=QUERY_MASK_SIDE):
    """Use official MAGFiLO expert-supplied spines, not estimated foreground skeletons."""
    center=np.zeros((target,target),np.uint8)
    scale=target/float(side)
    for a in annotations:
        spine=a.get('spine',[])
        if not isinstance(spine,list) or len(spine)<4 or len(spine)%2:continue
        pts=np.asarray(spine,np.float32).reshape(-1,2)
        pts=np.rint((pts-[x0,y0])*scale).astype(np.int32)
        if pts.shape[0]>=2:cv2.polylines(center,[pts],False,1,thickness=1)
    return center

class FilamentInstanceCrops(Dataset):
    def __init__(self,records,augment=False):self.records=records;self.augment=augment
    def __len__(self):return len(self.records)
    def __getitem__(self,index):
        stem,x0,y0,k=self.records[index]
        gray=cv2.imread(str(TRAIN[stem]),cv2.IMREAD_GRAYSCALE)
        if gray is None:raise FileNotFoundError(stem)
        roi=gray[y0:y0+QUERY_CROP_NATIVE,x0:x0+QUERY_CROP_NATIVE]
        versions=[ann for _,ann in VERSIONS[stem] if ann]
        # Deterministic expert choices for held validation; stochastic versions for training.
        version=versions[np.random.randint(len(versions)) if self.augment else k%len(versions)]
        masks=masks_in_crop(version,x0,y0,QUERY_CROP_NATIVE)
        expert_spines=annotated_spine_map(version,x0,y0,QUERY_CROP_NATIVE)
        im=cv2.resize(roi,(QUERY_SIDE,QUERY_SIDE),interpolation=cv2.INTER_AREA)
        if self.augment:
            if np.random.rand()<.5:im=np.fliplr(im).copy();masks=masks[:,:,::-1].copy();expert_spines=np.fliplr(expert_spines).copy()
            if np.random.rand()<.5:im=np.flipud(im).copy();masks=masks[:,::-1].copy();expert_spines=np.flipud(expert_spines).copy()
            if np.random.rand()<.5:
                rot=np.random.randint(4);im=np.rot90(im,rot).copy();masks=np.rot90(masks,rot,axes=(1,2)).copy();expert_spines=np.rot90(expert_spines,rot).copy()
            if np.random.rand()<.3:
                im=np.clip(im.astype(np.float32)*np.random.uniform(.85,1.18)+np.random.uniform(-12,12),0,255).astype(np.uint8)
        eq=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(im)
        inp=np.stack([im/255.,eq/255.],axis=0).astype(np.float32)
        fg=np.zeros((QUERY_MASK_SIDE,QUERY_MASK_SIDE),np.uint8)
        center=np.zeros_like(fg);edge=np.zeros_like(fg)
        for m in masks:
            fg|=m
            if m.sum()>4:
                # Use the official expert spine where available; fallback to a mask skeleton.
                edge|=(cv2.morphologyEx(m,cv2.MORPH_GRADIENT,np.ones((3,3),np.uint8))>0).astype(np.uint8)
        # Faint spines may rasterize to zero after downsampling: fallback conservatively.
        center=expert_spines if expert_spines.any() else skeletonize(fg>0).astype(np.uint8)
        center=(center & fg).astype(np.uint8)
        if not center.any() and fg.any():center=skeletonize(fg>0).astype(np.uint8)
        # Orientation target as tangent proxy of expert spine, supervised only near spine.
        smooth=cv2.GaussianBlur(center.astype(np.float32),(7,7),1.25)
        gx=cv2.Sobel(smooth,cv2.CV_32F,1,0,ksize=3);gy=cv2.Sobel(smooth,cv2.CV_32F,0,1,ksize=3)
        norm=np.sqrt(gx*gx+gy*gy)+1e-6
        orient=np.stack([-gy/norm,gx/norm],0).astype(np.float32)
        return torch.from_numpy(inp),torch.from_numpy(masks.astype(np.float32)),               torch.from_numpy(np.stack([center,edge],0).astype(np.float32)),torch.from_numpy(orient)

def instance_collate(batch):
    xs,ys,aux,oris=zip(*batch)
    return torch.stack(xs),list(ys),torch.stack(aux),torch.stack(oris)

train_loader=DataLoader(FilamentInstanceCrops(TRAIN_RECORDS,True),batch_size=BATCH,
    shuffle=True,num_workers=2,pin_memory=True,collate_fn=instance_collate)
val_loader=DataLoader(FilamentInstanceCrops(VAL_RECORDS,False),batch_size=BATCH,
    shuffle=False,num_workers=2,pin_memory=True,collate_fn=instance_collate)
# Semantic-union failure regression: two overlapping object masks remain separate.
toy_a=np.zeros((QUERY_MASK_SIDE,QUERY_MASK_SIDE),np.uint8)
toy_b=toy_a.copy();toy_a[14:40,12:35]=1;toy_b[22:49,28:45]=1
assert np.stack([toy_a,toy_b]).shape[0]==2


## 7. FilamentFormer-Lite: learned queries + multi-head topology decoder

A compact convolutional feature pyramid yields stride-4 pixel embeddings and stride-16 transformer memory. 24 learned object queries attend to multiscale context, independently emit instance masks, objectness and a predicted-IoU score. Side outputs supervise filament centerlines, local boundaries and orientation. This is **not** the official Mask2Former implementation; it is an intentionally smaller research adaptation designed for T4.

In [ ]:
class SmallConv(nn.Module):
    def __init__(self,ni,no,stride=1):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(ni,no,3,stride,1,bias=False),nn.BatchNorm2d(no),nn.GELU(),
              nn.Conv2d(no,no,3,1,1,bias=False),nn.BatchNorm2d(no),nn.GELU())
    def forward(self,x):return self.net(x)

class FilamentFormerLite(nn.Module):
    def __init__(self,queries=QUERY_COUNT,width=128):
        super().__init__();self.queries=queries
        self.stem=SmallConv(2,32,2)     # 1/2
        self.s4=SmallConv(32,64,2)      # 1/4
        self.s8=SmallConv(64,96,2)      # 1/8
        self.s16=SmallConv(96,160,2)    # 1/16
        self.pixel_fpn=nn.Sequential(nn.Conv2d(64+96+160,width,1),nn.GELU(),
                              nn.Conv2d(width,width,3,padding=1),nn.GELU())
        self.memory_proj=nn.Conv2d(160,width,1)
        self.query_embed=nn.Embedding(queries,width)
        self.query_pos=nn.Embedding(queries,width)
        self.decoder=nn.TransformerDecoder(nn.TransformerDecoderLayer(width,4,dim_feedforward=384,
                  dropout=.05,batch_first=True,activation='gelu',norm_first=True),num_layers=2)
        self.class_head=nn.Linear(width,1)
        self.mask_embed=nn.Linear(width,width)
        self.iou_head=nn.Sequential(nn.Linear(width,64),nn.GELU(),nn.Linear(64,1))
        self.struct_head=nn.Conv2d(width,4,1)  # centerline, boundary, tangent-x, tangent-y
        self._pos_cache={}
    def position_encoding(self,h,w,device,dtype):
        # Fourier 2D encoding: 128 features, stable across input tiles.
        y,x=torch.meshgrid(torch.linspace(0,1,h,device=device,dtype=dtype),
                           torch.linspace(0,1,w,device=device,dtype=dtype),indexing='ij')
        freqs=torch.exp(torch.arange(16,device=device,dtype=dtype)*(-math.log(10000.)/16.))
        pts=torch.stack([x[...,None]*freqs,y[...,None]*freqs],dim=-1)
        return torch.cat([pts[...,0].sin(),pts[...,0].cos(),pts[...,1].sin(),pts[...,1].cos()],dim=-1).reshape(1,h*w,64)
    def forward(self,x):
        a=self.stem(x);b=self.s4(a);c=self.s8(b);d=self.s16(c)
        sz=b.shape[-2:]
        pixel=self.pixel_fpn(torch.cat([b,F.interpolate(c,size=sz,mode='bilinear',align_corners=False),
                                       F.interpolate(d,size=sz,mode='bilinear',align_corners=False)],dim=1))
        mem=self.memory_proj(d).flatten(2).transpose(1,2)
        pos=self.position_encoding(d.shape[2],d.shape[3],mem.device,mem.dtype)
        # 64-dimensional position embedding adds to first 64 of 128 memory features.
        mem=torch.cat([mem[:,:,:64]+pos,mem[:,:,64:]],dim=-1)
        queries=(self.query_embed.weight+self.query_pos.weight)[None].expand(x.shape[0],-1,-1)
        q=self.decoder(queries,mem)
        mask=torch.einsum('bqc,bchw->bqhw',self.mask_embed(q),pixel)/math.sqrt(pixel.shape[1])
        return {'logits':self.class_head(q).squeeze(-1),'masks':mask,
                'iou':self.iou_head(q).squeeze(-1), 'struct':self.struct_head(pixel)}

# Test tensor dimensions without allocating 512² full training memory on CPU.
model_smoke=FilamentFormerLite()
with torch.no_grad():
    test_out=model_smoke(torch.randn(1,2,128,128))
assert test_out['masks'].shape==(1,QUERY_COUNT,32,32)
assert test_out['struct'].shape==(1,4,32,32)
assert test_out['logits'].shape==(1,QUERY_COUNT)
del model_smoke,test_out
print('PASS: trainable query masks and topology head output dimensions.')


## 8. Hungarian assignment, multi-expert instance loss and centerline/edge supervision

In [ ]:
def hungarian_indices(pred_logits,pred_masks,target_masks):
    n=int(target_masks.shape[0]);q=int(pred_masks.shape[0])
    if n==0:return np.asarray([],int),np.asarray([],int)
    with torch.no_grad():
        prob=pred_masks.sigmoid().flatten(1)
        true=target_masks.flatten(1)
        # Dense instance-wise Dice matching, independent of filament instance IDs.
        similarity=(2*prob@true.T+1)/(prob.sum(1,keepdim=True)+true.sum(1)[None]+1)
        obj=pred_logits.sigmoid()[:,None]
        cost=(1-similarity)*2.5+(.65*(1-obj))
        ii,jj=linear_sum_assignment(cost.float().cpu().numpy())
    return ii,jj

def safe_dice_loss(pr,gt):
    prob=pr.sigmoid();num=2*(prob*gt).sum((-2,-1))+1
    den=(prob+gt).sum((-2,-1))+1
    return (1-num/den).mean()

def query_instance_loss(outputs,target_list,aux,oris,structured=True):
    object_losses=[];mask_losses=[];quality_losses=[]
    for i,tar in enumerate(target_list):
        gt=tar.to(outputs['masks'].device,non_blocking=True)
        pm=outputs['masks'][i];logits=outputs['logits'][i]
        pi,gi=hungarian_indices(logits,pm,gt)
        tcls=torch.zeros_like(logits)
        if len(pi):
            tcls[torch.as_tensor(pi,device=logits.device,dtype=torch.long)]=1.
            matched=pm[pi];truth=gt[gi]
            bce=F.binary_cross_entropy_with_logits(matched,truth)
            dloss=safe_dice_loss(matched,truth)
            mask_losses.append(.35*bce+.65*dloss)
            with torch.no_grad():
                binary=(matched.sigmoid()>.5).float()
                intersection=(binary*truth).sum((-2,-1))
                union=binary.sum((-2,-1))+truth.sum((-2,-1))-intersection
                target_iou=intersection/(union+1e-6)
            quality_losses.append(F.smooth_l1_loss(outputs['iou'][i,pi].sigmoid(),target_iou))
        positive_weight=torch.where(tcls>0,torch.full_like(tcls,3.),torch.ones_like(tcls))
        object_losses.append(F.binary_cross_entropy_with_logits(logits,tcls,weight=positive_weight))
    objective=torch.stack(object_losses).mean()
    if mask_losses:objective=objective+3.0*torch.stack(mask_losses).mean()
    if quality_losses:objective=objective+.4*torch.stack(quality_losses).mean()
    if structured:
        struct=outputs['struct'];aux=aux.to(struct.device,non_blocking=True)
        # Positive-weighted centerline, boundary and structural Dice.
        line=struct[:,0:1];edge=struct[:,1:2]
        center_gt=aux[:,0:1];boundary_gt=aux[:,1:2]
        center_loss=F.binary_cross_entropy_with_logits(line,center_gt,pos_weight=torch.tensor(10.,device=line.device))
        edge_loss=F.binary_cross_entropy_with_logits(edge,boundary_gt,pos_weight=torch.tensor(4.,device=edge.device))
        center_loss=center_loss+safe_dice_loss(line,center_gt)
        orient_pred=F.normalize(struct[:,2:4],dim=1)
        target_ori=oris.to(struct.device,non_blocking=True)
        orientation_mask=(center_gt>0).float()
        orient_loss=((1-(orient_pred*target_ori).sum(1,keepdim=True).abs())*orientation_mask).sum()/(orientation_mask.sum()+1)
        objective=objective+.20*center_loss+.12*edge_loss+.035*orient_loss
    return objective

# Forward + backward on small crops before committing to T4 training.
probe=FilamentFormerLite(queries=8)
xx=torch.randn(1,2,64,64)
yy=torch.zeros((1,16,16));yy[:,2:8,3:10]=1.
pred=probe(xx)
loss=query_instance_loss(pred,[yy],torch.zeros(1,2,16,16),torch.zeros(1,2,16,16),True)
loss.backward()
assert math.isfinite(float(loss)) and any(p.grad is not None for p in probe.parameters())
del probe,xx,yy,pred,loss
print('PASS: differentiable Hungarian-matched loss / gradient propagation.')


## 9. Checkpointed training of full structure-supervised and plain-query ablation

In [ ]:
import gc
DEVICE=torch.device('cuda:'+str(GPU_ID))
assert torch.cuda.is_available(),'Enable Kaggle GPU accelerator'
torch.backends.cudnn.benchmark=True

# Avoid overwriting completed checkpoints on re-run. Saved model archives can be attached later.
def restore_checkpoint(name):
    destination=Path('/kaggle/working')/name
    if destination.is_file():return destination
    payload,source=find_original_asset(name)
    if payload is not None:
        destination.write_bytes(payload);print('Restored previously trained',name,'from',source)
    return destination

def train_query_model(tag,structure,epochs,steps):
    best=restore_checkpoint('SOLAR_V27_'+tag+'_BEST.pt')
    last=Path('/kaggle/working/SOLAR_V27_'+tag+'_LAST.pt')
    network=FilamentFormerLite().to(DEVICE)
    opt=torch.optim.AdamW(network.parameters(),lr=TRAINING_LR,weight_decay=.0002)
    scaler=torch.amp.GradScaler('cuda')
    start=0;best_loss=float('inf')
    if best.is_file():
        prev=torch.load(best,map_location='cpu',weights_only=False)
        network.load_state_dict(prev['model']);start=int(prev.get('epoch',-1))+1
        best_loss=float(prev.get('val_loss',float('inf')))
    history=[]
    for ep in range(start,epochs):
        network.train();running=[]
        for j,(x,t,aux,ori) in enumerate(train_loader):
            if j>=steps:break
            x=x.to(DEVICE,non_blocking=True);opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                output=network(x);err=query_instance_loss(output,t,aux,ori,structured=structure)
            scaler.scale(err).backward();scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(network.parameters(),1.)
            scaler.step(opt);scaler.update();running.append(float(err.detach()))
        network.eval();values=[]
        with torch.inference_mode():
            for j,(x,t,aux,ori) in enumerate(val_loader):
                if j>=55:break
                x=x.to(DEVICE,non_blocking=True)
                with torch.autocast('cuda',dtype=torch.float16):
                    err=query_instance_loss(network(x),t,aux,ori,structured=structure)
                values.append(float(err))
        score=float(np.mean(values)) if values else float('inf')
        checkpoint={'epoch':ep,'val_loss':score,'structure':structure,
             'model':{k:v.detach().cpu() for k,v in network.state_dict().items()},
             'dataset_train_scenes':len(TRAIN_INNER),'train_test_label_overlap':False,
             'architecture':'FilamentFormerLite-ConvFPN-Query24'}
        torch.save(checkpoint,last)
        if score<best_loss:
            best_loss=score
            temporary=best.with_suffix('.tmp');torch.save(checkpoint,temporary);temporary.replace(best)
        history.append({'epoch':ep,'train_loss':float(np.mean(running)),
               'inner_val_loss':score,'best_loss':best_loss,'structure_supervision':structure})
        print(tag,'epoch',ep,'train',round(float(np.mean(running)),4),
              'inner_val',round(score,4),'best',round(best_loss,4),flush=True)
        pd.DataFrame(history).to_csv(OUT/'reports'/('V27_'+tag+'_training.csv'),index=False)
    assert best.is_file(),('No checkpoint generated',best)
    package=torch.load(best,map_location='cpu',weights_only=False)
    network.load_state_dict(package['model'],strict=True);network.eval()
    print('Query model',tag,'best epoch',package['epoch'],'SHA',sha(best)[:18])
    return network,best

query_full,QUERY_FULL_PATH=train_query_model('STRUCTURE',True,EPOCHS,STEPS_PER_EPOCH)
if TRAIN_PLAIN_ABLATION:
    query_plain,QUERY_PLAIN_PATH=train_query_model('PLAIN',False,PLAIN_EPOCHS,PLAIN_STEPS)
else:
    query_plain=None;QUERY_PLAIN_PATH=None
print('All requested trained query checkpoints available; V25 detector/refiners remain frozen.')


## 10. High-resolution tiled independent-query inference

The query model predicts individual instance masks on each 768px native crop; masks are mapped to the original 2048px scene and deduplicated. Internal tile-boundary fragments are rejected to avoid turning one filament into multiple false positives. We cache the raw query outputs per scene/model SHA so interrupted Kaggle runs can continue.

In [ ]:
def query_windows():
    coords=list(range(0,H-QUERY_CROP_NATIVE+1,WINDOW_STRIDE))
    if coords[-1]!=H-QUERY_CROP_NATIVE:coords.append(H-QUERY_CROP_NATIVE)
    return [(x,y) for y in coords for x in coords]
WINDOWS=query_windows()
assert len(WINDOWS)==9

@torch.inference_mode()
def query_proposals(stem,network,weight_path,conf=RAW_QUERY_CONF):
    weight_signature=sha(weight_path)
    cached=OUT/'query_cache'/(weight_signature[:16]+'_'+stem+'.json.gz')
    sigval=hashlib.sha256(json.dumps({'model':weight_signature,'stem':stem,'windows':WINDOWS,
                    'prob':QUERY_MASK_PROB,'conf':conf,'seam':QUERY_TILE_SEAM_GUARD,
                    'area':MIN_QUERY_AREA},sort_keys=True).encode()).hexdigest()
    if cached.exists():
        try:
            with gzip.open(cached,'rt') as f:v=json.load(f)
            if v.get('sig')==sigval:return v['items']
        except (ValueError,OSError,KeyError):pass
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    assert gray is not None and gray.shape==(H,W)
    found=[]
    network.eval()
    for x0,y0 in WINDOWS:
        roi=gray[y0:y0+QUERY_CROP_NATIVE,x0:x0+QUERY_CROP_NATIVE]
        img=cv2.resize(roi,(QUERY_SIDE,QUERY_SIDE),interpolation=cv2.INTER_AREA)
        equ=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(img)
        tensor=torch.from_numpy(np.stack([img/255.,equ/255.],0).astype(np.float32))[None].to(DEVICE)
        with torch.autocast('cuda',dtype=torch.float16):out=network(tensor)
        confidence=(out['logits'].sigmoid()*(.65+.35*out['iou'].sigmoid()))[0].float().cpu().numpy()
        segmentation=out['masks'][0].sigmoid().float().cpu().numpy()
        center=out['struct'][0,0].sigmoid().float().cpu().numpy()
        for j,score in enumerate(confidence):
            if score<conf:continue
            little=(segmentation[j]>QUERY_MASK_PROB).astype(np.uint8)
            if int(little.sum())<8:continue
            mask=cv2.resize(little,(QUERY_CROP_NATIVE,QUERY_CROP_NATIVE),interpolation=cv2.INTER_NEAREST)
            area=int(mask.sum())
            if area<MIN_QUERY_AREA:continue
            yy,xx=np.where(mask)
            if QUERY_TILE_SEAM_GUARD:
                if ((x0>0 and xx.min()<8) or (x0+QUERY_CROP_NATIVE<W and xx.max()>=QUERY_CROP_NATIVE-8)
                    or (y0>0 and yy.min()<8) or (y0+QUERY_CROP_NATIVE<H and yy.max()>=QUERY_CROP_NATIVE-8)):
                    continue
            csmall=(cv2.resize(center,(QUERY_CROP_NATIVE,QUERY_CROP_NATIVE),interpolation=cv2.INTER_LINEAR)>0.45)
            center_support=float(np.mean(csmall[mask>0]))
            full=np.zeros((H,W),np.uint8);full[y0:y0+QUERY_CROP_NATIVE,x0:x0+QUERY_CROP_NATIVE]=mask
            rr=enc(full)
            found.append({'counts':txt(rr),'score':float(score),'center_support':center_support,
                          'area':area,'window':(x0,y0),'query':j})
    found.sort(key=lambda p:(-p['score'],-p['area']))
    distinct=[];bank=[]
    for candidate in found:
        rr=as_rle(candidate['counts'])
        if bank and float(np.max(coco.iou([rr],bank,[0]*len(bank))))>=QUERY_NMS_IOU:continue
        distinct.append(candidate);bank.append(rr)
        if len(distinct)>=QUERY_MAX_PER_SCENE:break
    cached.parent.mkdir(parents=True,exist_ok=True)
    with gzip.open(cached,'wt',compresslevel=3) as f:
        json.dump({'sig':sigval,'items':distinct},f,separators=(',',':'))
    return distinct

print('Query tiling ready:',len(WINDOWS),'overlapping windows/scene')


## 11. Train an out-of-fold quality gate for new V27 instances, without retraining V25

A fresh gate decides whether a **novel V27 query mask** is likely to match an expert instance at IoU > 0.5; it must add true positives without overwhelming V25's existing masks. The gate fits on historical calibration days only, chooses a threshold by grouped OOF, and is tested on disjoint audit days. It does *not* train on hidden test masks.

In [ ]:
def append_novel(old_masks,new_masks,minfree=FUSION_MIN_FREE,nms=FUSION_NMS_IOU):
    # Preserve old V25 mask bytes/order; candidate quality filtering happens upstream.
    originals=list(old_masks)
    occupied=np.zeros((H,W),np.uint8)
    for rr in old_masks:occupied|=coco.decode(rr).astype(np.uint8)
    accepted=[];bank=list(old_masks)
    for rr in new_masks:
        if bank:
            overlapping=float(np.max(coco.iou([rr],bank,[0]*len(bank))))
            if overlapping>=nms:continue
        mask=coco.decode(rr).astype(np.uint8)
        free=mask&(occupied==0)
        if int(free.sum())<MIN_QUERY_AREA or free.sum()/max(1,mask.sum())<minfree:continue
        new=enc(free);accepted.append(new);occupied|=free;bank.append(new)
    return originals+accepted


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

# All quality inputs are available from images and predictions, never hidden test ground truth.
QFEATURES=['query_conf','query_center_support','log_area','log_width','log_height',
    'aspect_log','extent','darkness','inside_std','nearby_v25_iou','nearby_v25_distance',
    'global_iou','tile_iou','rim_location','skeleton_density']

def query_feature_table(stem,proposals,v25_masks):
    if not proposals:return pd.DataFrame(columns=QFEATURES)
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    raw={SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    existing=np.asarray([coco.toBbox(rr) for rr in v25_masks],float).reshape(-1,4)
    rows=[]
    for prop in proposals:
        rr=as_rle(prop['counts']);x,y,w,h=map(float,coco.toBbox(rr))
        pix=coco.decode(rr).astype(np.uint8)
        yy,xx=np.where(pix)
        if not len(xx):continue
        values=gray[yy,xx].astype(float)
        xa=max(0,int(x)-10);ya=max(0,int(y)-10)
        xb=min(W,int(x+w)+10);yb=min(H,int(y+h)+10)
        mask=pix[ya:yb,xa:xb]
        region=gray[ya:yb,xa:xb]
        ring=cv2.dilate(mask,np.ones((7,7),np.uint8))&(mask==0)
        outer=float(np.median(region[ring>0])) if ring.any() else float(np.mean(values))
        querybox=np.array([x+w/2,y+h/2])
        centroids=existing[:,:2]+existing[:,2:]/2 if len(existing) else np.zeros((0,2))
        ndist=float(np.linalg.norm(centroids-querybox,axis=1).min()/2048) if len(existing) else 1.
        nearest=float(np.max(coco.iou([rr],v25_masks,[0]*len(v25_masks)))) if v25_masks else 0.
        gi=float(np.max(coco.iou([rr],[as_rle(p['counts']) for p in raw[SRC_HI]],
                       [0]*len(raw[SRC_HI])))) if raw[SRC_HI] else 0.
        ti=float(np.max(coco.iou([rr],[as_rle(p['counts']) for p in raw[SRC_TILES]],
                       [0]*len(raw[SRC_TILES])))) if raw[SRC_TILES] else 0.
        local=pix[int(y):min(H,int(y+h)+1),int(x):min(W,int(x+w)+1)]
        small=cv2.resize(local,(min(96,local.shape[1]),min(96,local.shape[0])),interpolation=cv2.INTER_NEAREST)
        skel=float(skeletonize(small>0).sum()/max(1,small.sum()))
        r=np.hypot(x+w/2-W/2,y+h/2-H/2)/(W/2)
        rows.append(dict(query_conf=prop['score'],query_center_support=prop['center_support'],
           log_area=np.log1p(prop['area']),log_width=np.log1p(w),log_height=np.log1p(h),
           aspect_log=np.log((w+1)/(h+1)),extent=prop['area']/max(1,w*h),
           darkness=(outer-np.mean(values))/255.,inside_std=float(values.std())/255.,
           nearby_v25_iou=nearest,nearby_v25_distance=ndist,
           global_iou=gi,tile_iou=ti,rim_location=r,skeleton_density=skel))
    df=pd.DataFrame(rows,columns=QFEATURES).replace([np.inf,-np.inf],0).fillna(0.)
    assert len(df)==len(proposals)
    return df

def query_true_labels(stem,props):
    result=[]
    for p in props:
        rr=as_rle(p['counts']);votes=[]
        for expert in gt(stem):
            overlaps=np.asarray(coco.iou([rr],expert,[0]*len(expert)),float).reshape(-1) if expert else np.asarray([])
            votes.append(float(overlaps.max()) if len(overlaps) else 0.)
        result.append(int(np.mean(np.asarray(votes)>.5)>=.5) if votes else 0)
    return result

def make_quality():
    return HistGradientBoostingClassifier(max_iter=120,max_leaf_nodes=9,
                  learning_rate=.045,min_samples_leaf=12,l2_regularization=6.,random_state=SEED+27)

QPATH=Path('/kaggle/working/SOLAR_V27_QUERY_QUALITY.joblib')
# Capture V25 and new query masks in acquisition-day grouped calibration scenes.
def build_gate_bank(network,path,tag):
    frames=[];truth=[];groups=[];lengths={};records={}
    for stem in tqdm(CAL,desc='Query calibration '+tag):
        old=frozen_v25_masks(stem)
        props=query_proposals(stem,network,path)
        feats=query_feature_table(stem,props,old)
        labels=query_true_labels(stem,props)
        assert len(feats)==len(labels)
        frames.append(feats);truth+=labels;groups += [stem[:8]]*len(labels)
        records[stem]=(props,old);lengths[stem]=len(labels)
    x=pd.concat(frames,ignore_index=True).reindex(columns=QFEATURES).fillna(0.)
    return x,np.asarray(truth,int),np.asarray(groups),lengths,records

QX,QY,QGROUP,Q_LENGTHS,CAL_RECORDS=build_gate_bank(query_full,QUERY_FULL_PATH,'STRUCTURE')
print('New-instance calibration:',len(QY),'candidate masks, true-positive fraction',float(QY.mean()) if len(QY) else None)
QGATE_ACTIVE=bool(len(QY)>=80 and QY.sum()>=12 and (len(QY)-QY.sum())>=25 and len(set(QGROUP))>=4)
if QGATE_ACTIVE:
    folds=GroupKFold(n_splits=4);oof=np.full(len(QY),np.nan)
    for tr,va in folds.split(QX,QY,QGROUP):
        if len(set(QY[tr]))<2:QGATE_ACTIVE=False;break
        learner=make_quality();learner.fit(QX.iloc[tr],QY[tr]);oof[va]=learner.predict_proba(QX.iloc[va])[:,1]
if QGATE_ACTIVE:
    offset=0;layout={}
    for stem in CAL:
        n=Q_LENGTHS[stem];layout[stem]=(offset,offset+n);offset+=n
    def choose_on_cal(threshold):
        count=np.zeros(4,float)
        for stem in CAL:
            p,old=CAL_RECORDS[stem];a,b=layout[stem]
            extras=[as_rle(pp['counts']) for pp,z in zip(p,oof[a:b]) if z>=threshold]
            candidate=append_novel(old,extras)
            count+=physical_counts(stem,candidate)
        return pq(count)
    baseline_cal=np.zeros(4,float)
    for stem in CAL:baseline_cal+=physical_counts(stem,CAL_RECORDS[stem][1])
    baseline_cal=pq(baseline_cal)
    oof_results={th:choose_on_cal(th) for th in QUALITY_CANDIDATE_THRESHOLDS}
    QTHRESHOLD=max(oof_results,key=lambda a:(oof_results[a],a))
    if oof_results[QTHRESHOLD]<baseline_cal+.0005:
        QGATE_ACTIVE=False
    else:
        qmodel=make_quality().fit(QX,QY)
        joblib.dump({'model':qmodel,'features':QFEATURES,'threshold':QTHRESHOLD,
             'oof_auc':float(roc_auc_score(QY,oof)),'calibration_days_only':True,
             'test_ground_truth_used':False},QPATH)
if not QGATE_ACTIVE:
    QTHRESHOLD=1.01;qmodel=None;oof_results={}
    print('No calibration evidence of novel-query improvement. Query-quality fusion disabled.')
else:
    print('New query quality cutoff',QTHRESHOLD,'CAL OOF PQ',oof_results)
save_json(OUT/'reports'/'V27_quality_training.json',{'active':QGATE_ACTIVE,
      'examples':len(QY),'positive':int(QY.sum()),'threshold':QTHRESHOLD,
      'oof_pq':oof_results,'calibration_days':len(CAL)})


## 12. Date-disjoint audit: query standalone, structured-vs-plain, V25 fusion

Compare one-to-one expert-PQ on audit scenes, including instance TP/FP/FN and distinct-day bootstrap. The gate is trained only on historical calibration days; the audit comes from different acquisition days, though earlier V25 research already reused the 104 scenes. V25 is never overwritten by unverified fusion.

In [ ]:
def select_query_candidates(stem,network,path,quality='learned',extra_margin=0.,old_masks=None):
    old=old_masks if old_masks is not None else frozen_v25_masks(stem)
    props=query_proposals(stem,network,path)
    if not props:return []
    if quality=='learned' and QGATE_ACTIVE:
        feats=query_feature_table(stem,props,old)
        p=qmodel.predict_proba(feats[QFEATURES])[:,1]
        return [as_rle(item['counts']) for item,score in zip(props,p) if score>=QTHRESHOLD+extra_margin]
    elif quality=='conf':return [as_rle(item['counts']) for item in props if item['score']>=.64+extra_margin]
    return []

METHODS=['V25_PROVEN_039','V27_QUERY_ALONE','V27_STRUCTURE_FUSION','V27_STRUCTURE_CONSERVATIVE',
         'V27_STRUCTURE_CONF_FUSION']
if query_plain is not None:METHODS += ['V27_PLAIN_FUSION']

def method_masks(stem,name,old=None):
    old=old if old is not None else frozen_v25_masks(stem)
    if name=='V25_PROVEN_039':return old
    if name=='V27_QUERY_ALONE':
        props=query_proposals(stem,query_full,QUERY_FULL_PATH)
        # Quality + non-overlap even for the standalone diagnostic, with no V25 additions.
        if QGATE_ACTIVE:
            feats=query_feature_table(stem,props,old)
            p=qmodel.predict_proba(feats[QFEATURES])[:,1] if len(feats) else []
            found=[as_rle(x['counts']) for x,z in zip(props,p) if z>=QTHRESHOLD]
        else:found=[as_rle(x['counts']) for x in props if x['score']>=.72]
        return append_novel([],found,minfree=.75,nms=.32)
    if name=='V27_STRUCTURE_FUSION':return append_novel(old,select_query_candidates(stem,query_full,QUERY_FULL_PATH,old_masks=old))
    if name=='V27_STRUCTURE_CONSERVATIVE':return append_novel(old,select_query_candidates(stem,query_full,QUERY_FULL_PATH,extra_margin=.12,old_masks=old),minfree=.82,nms=.15)
    if name=='V27_STRUCTURE_CONF_FUSION':return append_novel(old,select_query_candidates(stem,query_full,QUERY_FULL_PATH,quality='conf',old_masks=old))
    if name=='V27_PLAIN_FUSION':return append_novel(old,select_query_candidates(stem,query_plain,QUERY_PLAIN_PATH,quality='conf',old_masks=old))
    raise ValueError(name)

results={name:np.zeros(4,float) for name in METHODS};detail=[]
for stem in tqdm(AUDIT,desc='V27 audit on distinct acquisition days'):
    old=frozen_v25_masks(stem)
    for name in METHODS:
        masks=method_masks(stem,name,old)
        c=physical_counts(stem,masks)
        results[name]+=c
        detail.append({'stem':stem,'day':stem[:8],'method':name,'sum_iou':float(c[0]),
                       'TP':int(c[1]),'FP':int(c[2]),'FN':int(c[3]),'instances':len(masks)})
control='V25_PROVEN_039';base_score=pq(results[control])
board=pd.DataFrame([{'method':name,**metrics(c),'delta_vs_V25':pq(c)-base_score} for name,c in results.items()]).sort_values('PQ',ascending=False)
board.to_csv(OUT/'reports'/'V27_AUDIT_PQ.csv',index=False)
pd.DataFrame(detail).to_csv(OUT/'reports'/'V27_AUDIT_PER_SCENE.csv',index=False)
display(board)
assert abs(base_score-.458880)<.032,('Frozen V25 audit parity mismatch',base_score)

pair_day=pd.DataFrame(detail).groupby(['method','day'])[['sum_iou','TP','FP','FN']].sum()
days=sorted(set(item['day'] for item in detail))
arrays={name:np.stack([pair_day.loc[(name,d)].to_numpy(float) for d in days]) for name in METHODS}
bootindex=np.random.default_rng(SEED+27).integers(0,len(days),size=(1500,len(days)))
def vector_pq(z):
    den=z[:,1]+.5*(z[:,2]+z[:,3]);return np.divide(z[:,0],den,out=np.zeros(len(z)),where=den>0)
baseboot=vector_pq(arrays[control][bootindex].sum(axis=1))
boot=[]
for name in METHODS:
    if name==control:continue
    diff=vector_pq(arrays[name][bootindex].sum(axis=1))-baseboot
    boot.append({'method':name,'audit_delta':pq(results[name])-base_score,
        'boot_2p5':float(np.quantile(diff,.025)),'boot_97p5':float(np.quantile(diff,.975)),
        'positive_fraction':float((diff>0).mean())})
boot_df=pd.DataFrame(boot).sort_values('audit_delta',ascending=False)
boot_df.to_csv(OUT/'reports'/'V27_PAIRED_DAY_BOOTSTRAP.csv',index=False)
display(boot_df)
eligible=boot_df[(boot_df.audit_delta>=MIN_LOCAL_GAIN_TO_PROMOTE)&(boot_df.positive_fraction>=.90)]
PREFERRED=control if eligible.empty else eligible.iloc[0].method
print('V27 selected default:',PREFERRED,'— confirmed public best still 0.39 until graded')
save_json(OUT/'reports'/'V27_selection.json',{'recommended_default':PREFERRED,
  'public_confirmed_v25':.39,'target_not_promised':.40,'audit_PQ':{x:pq(y) for x,y in results.items()},
  'historical_holdout_reused':True,'query_quality_gate':QGATE_ACTIVE,'test_labels_accessed':False})


## 13. Full 180-test inference, exact 0.39 historical CSV protection, all experiment exports

**The original V25 file is preferred when its SHA prefix matches the previously scored `ac470fbf...`.** A regenerated V25 CSV may differ byte-for-byte; this notebook therefore never labels it the historically verified file unless the SHA matches. New V27 submissions always have independent file names.

In [ ]:
import csv
TEST_STEMS=sorted(TEST);assert len(TEST_STEMS)==180
EXACT_039=Path('/kaggle/working/V27_PROVEN_V25_PUBLIC_039.csv')
confirmed_bytes=None;source_exact=None
for filename in ('V25_REAL_REFINE_PLUS_QUALITY.csv','submission.csv'):
    content,origin=find_original_asset(filename)
    if content is not None and hashlib.sha256(content).hexdigest().startswith(V25_EXPECTED_CSV_SHA):
        confirmed_bytes=content;source_exact=origin;break
if confirmed_bytes is not None:
    EXACT_039.write_bytes(confirmed_bytes)
    assert len(pd.read_csv(EXACT_039))==V25_EXPECTED_ROWS
    print('PROTECTED: exact 0.39 Kaggle submission bytes recovered from',source_exact)
else:
    print('Historical .39 CSV not attached; attempt regeneration, but require exact SHA before claiming byte parity.')

# Inference caches are resumable; models are checkpointed *before* this stage.
for stem in tqdm(TEST_STEMS,desc='V27 test image query predictions'):
    query_proposals(stem,query_full,QUERY_FULL_PATH)
    if query_plain is not None:query_proposals(stem,query_plain,QUERY_PLAIN_PATH)

csv_paths={name:OUT/'submissions'/(name+'.csv') for name in METHODS}
rows=[]
from contextlib import ExitStack
with ExitStack() as stack:
    writers={name:csv.writer(stack.enter_context(open(p,'w',newline='',encoding='utf8'))) for name,p in csv_paths.items()}
    for wr in writers.values():wr.writerow(['filament_id','segmentation_rle'])
    for stem in tqdm(TEST_STEMS,desc='Export V27 instances'):
        old=frozen_v25_masks(stem)
        for name,writer in writers.items():
            masks=method_masks(stem,name,old)
            rows.append({'stem':stem,'method':name,'instances':len(masks)})
            for idx,rr in enumerate(masks,1):writer.writerow([f'{stem}_{idx}',txt(rr)])
countframe=pd.DataFrame(rows)
countframe.to_csv(OUT/'reports'/'V27_TEST_INSTANCE_COUNTS.csv',index=False)
files={}
for name,path in csv_paths.items():
    df=pd.read_csv(path,dtype=str)
    assert df.columns.tolist()==['filament_id','segmentation_rle'] and df.filament_id.is_unique
    assert df.segmentation_rle.notna().all()
    assert df.filament_id.str.rsplit('_',n=1).str[0].isin(TEST).all()
    assert len(df)==int(countframe[countframe.method==name].instances.sum())
    for encoded in df.segmentation_rle:assert int(coco.area(as_rle(encoded)))>0
    files[name]={'path':str(path),'sha256':sha(path),'rows':len(df)}
    print(name,len(df),'SHA:',files[name]['sha256'][:20])

control_path=csv_paths[control]
if not EXACT_039.is_file() and sha(control_path).startswith(V25_EXPECTED_CSV_SHA):
    shutil.copy2(control_path,EXACT_039)
assert EXACT_039.is_file() and sha(EXACT_039).startswith(V25_EXPECTED_CSV_SHA),(
    'Could not recover exact V25 public-0.39 bytes. Attach SOLAR_V25_ALL_PREDICTIONS_AND_REPORTS.zip '
    'or the original V25_REAL_REFINE_PLUS_QUALITY.csv to preserve a proven baseline.')

SUBMISSION=Path('/kaggle/working/submission.csv')
source=EXACT_039 if PREFERRED==control else csv_paths[PREFERRED]
shutil.copy2(source,SUBMISSION)
for index,name in enumerate([n for n in METHODS if n!=control],1):
    shutil.copy2(csv_paths[name],Path('/kaggle/working')/f'V27_ALT_{index}.csv')
save_json(OUT/'reports'/'V27_manifest.json',{'verified_public_pq':.39,'target_public_pq':.40,
     'default_method':PREFERRED,'baseline_historical_sha':sha(EXACT_039),
     'model_checkpoint_structure_sha':sha(QUERY_FULL_PATH),
     'model_checkpoint_plain_sha':sha(QUERY_PLAIN_PATH) if QUERY_PLAIN_PATH else None,
     'new_public_score':None,'query_quality_gate':QGATE_ACTIVE,'test_labels_accessed':False,
     'candidate_files':files})
print('DEFAULT:',PREFERRED, '| File:',SUBMISSION, '| Verified old reference:',EXACT_039)


## 14. Final model bundles, reproducibility manifest, and run interpretation

In [ ]:
REPORTS_ZIP=Path('/kaggle/working/SOLAR_V27_PREDICTIONS_AND_REPORTS.zip')
with zipfile.ZipFile(REPORTS_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3,allowZip64=True) as z:
    for p in sorted((OUT/'reports').glob('V27_*.csv'))+sorted((OUT/'reports').glob('V27_*.json')):
        z.write(p,arcname='reports/'+p.name)
    for p in sorted((OUT/'submissions').glob('*.csv')):z.write(p,arcname='submissions/'+p.name)
    z.write(SUBMISSION,'submission.csv');z.write(EXACT_039,EXACT_039.name)
with zipfile.ZipFile(REPORTS_ZIP) as z:assert z.testzip() is None

MODELS_ZIP=Path('/kaggle/working/SOLAR_V27_ALL_TRAINED_CHECKPOINTS.zip')
with zipfile.ZipFile(MODELS_ZIP,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    for p in WEIGHTS.values():z.write(p,arcname='detectors/'+p.name)
    for p in (V24_REFINER_PATH,V24_QUALITY_PATH,V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH,QUERY_FULL_PATH,QUERY_PLAIN_PATH,QPATH):
        if p is not None and Path(p).is_file():z.write(p,arcname='models/'+Path(p).name)
    z.write(REPORTS_ZIP,arcname=REPORTS_ZIP.name)
with zipfile.ZipFile(MODELS_ZIP) as z:assert z.testzip() is None
print('ALL FINISHED. Public best confirmed: 0.39, V27 score unverified.')
print('Reports:',REPORTS_ZIP,round(REPORTS_ZIP.stat().st_size/2**20,2),'MiB')
print('Model checkpoints:',MODELS_ZIP,round(MODELS_ZIP.stat().st_size/2**20,2),'MiB')
print('Review V27_AUDIT_PQ.csv, V27_PAIRED_DAY_BOOTSTRAP.csv and V27_selection.json.')


### Notes for interpreting V27

- **The query model is new; V25 remains frozen.** V25 supplies the 0.39 control; V27 never uses hidden test annotations.
- **Instance queries differ from the old curve U-Net:** each of 24 learned queries predicts its own filament mask and objectness rather than creating connected components from a single semantic foreground map.
- **Controlled experiments:** standalone query segmentation, learned-quality fusion, conservative fusion, confidence-only fusion, and an optional plain-query ablation are exported separately. An option failing date-disjoint audit does *not* automatically replace the 0.39 winner.
- **Limits:** Original 603/104 split derives from earlier work; cross-fitting a fresh detector or using genuinely untouched observations would be necessary to make more defensible generalization claims. The current in-notebook evaluator is a historical proxy, not proof of the hidden test PQ.
- **Resume:** Save the full model ZIP or attach the new `.pt` checkpoints to a future Kaggle run; new queries and test proposal caches can also be saved as a Kaggle output dataset to avoid recomputing.
- **GPU budget:** `TRAIN_PLAIN_ABLATION=False` is a useful first switch to shorten the run while retaining the primary structured-query experiment.